# INFOSCI 301 Week 4 Network Idiom Studio

**Real data:** Zachary's Karate Club network and a reproducible OpenAlex scholarly-metadata snapshot.  
**Question:** Which network idiom and algorithm make the user's task possible without turning structure into decoration?  
**Final-project addition:** one network design decision, one output, one evidence boundary, and one next test.

This notebook follows **Domain -> Data and Task -> Idiom -> Algorithm (validate each level)**. Run all cells. The bundled snapshot works offline; the optional API-refresh cell is clearly marked.

## Read, explore, revise, and explain / 阅读、探索、修改与解释

Follow **Domain → Data/Task → Idiom → Algorithm**, and validate each decision.
从领域与人群出发，依次说明数据与任务、视觉表达、算法，并逐层检验。

1. **Core route:** inspect the bundled evidence, run a baseline, compare the named idioms, then change one meaningful parameter while keeping the other records and scales fixed.
2. **Optional depth:** specialist cells retain their original source. Set `RUN_PACKAGE_EXTENSIONS=True` and install the documented package only when your question needs it. A skipped cell is not an executed algorithm.
3. **Your contribution:** explain three human design decisions, one supported reading, one limitation, and the next test. Peer-feedback fields remain empty until someone actually tests the artifact.

Vercel displays a **saved execution generated from this notebook**. Python widgets require a Colab session. Static figures preserve the evidence when widgets or GPU views are unavailable. Package installation and external browser renderers may need internet; the core lesson data require no live API or key.
Vercel 展示由本笔记本生成的已保存结果。Python 控件需在 Colab 中运行；静态图保留相同数据与证据边界。

## Definition of done

1. State a user and task: find, compare, trace, identify clusters, or explain change.
2. Build a node-edge table from a real dataset.
3. Compare at least two layouts and two structural measures.
4. Produce one static and one interactive network view.
5. State what the view supports, what it does not establish, and what you will add to the group MVP.

In [ ]:
# Install only missing core libraries. Data are bundled; package downloads may need internet.
import importlib.util,subprocess,sys
RUN_PACKAGE_EXTENSIONS=False
_extension_status={}
requirements={'pandas': 'pandas', 'numpy': 'numpy', 'matplotlib': 'matplotlib', 'plotly': 'plotly', 'networkx': 'networkx', 'sklearn': 'scikit-learn'}
missing=[pkg for module,pkg in requirements.items() if importlib.util.find_spec(module) is None]
if missing:subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
print('Core environment ready; no live data API or key is required.')

### Shared plotting style

| Decision / 决策 | This cell / 本单元 |
|---|---|
| Idiom / 视觉表达 | Shared plotting style |
| Task / 任务 | Inspect, compare, or explain the stated evidence. |
| Algorithm / 算法 | Inspect the named implementation and parameters in the following cell. |
| Data and units / 数据与单位 | Use the adjacent source note, units, sample, exclusions, and baseline. |
| Evidence boundary / 证据边界 | A display does not establish causality, consent, or community value. |

**Try and explain / 尝试并解释:** keep the baseline, change one documented parameter, and explain which reading changes. Retain the original and revised outputs.

In [ ]:
from pathlib import Path
from collections import Counter
import base64, gzip, io, json, math, textwrap, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import plotly.graph_objects as go
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import HTML, display

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 120, "font.size": 10, "axes.titlesize": 12})
OUTPUT = Path("outputs")
OUTPUT.mkdir(exist_ok=True)
NAVY, TEAL, PURPLE, GOLD, CORAL = "#173F4A", "#21747A", "#6A5983", "#C59A43", "#C76D5A"

### App-matched paper–topic comparison

| Decision / 决策 | This cell / 本单元 |
|---|---|
| Idiom / 视觉表达 | App-matched paper–topic comparison |
| Task / 任务 | Compare identical records across views. |
| Algorithm / 算法 | Inspect the named implementation and parameters in the following cell. |
| Data and units / 数据与单位 | Use the adjacent source note, units, sample, exclusions, and baseline. |
| Evidence boundary / 证据边界 | Topic assignment is not citation; designation is not construction or cultural importance. |

**Try and explain / 尝试并解释:** keep the baseline, change one documented parameter, and explain which reading changes. Retain the original and revised outputs.

In [ ]:
import base64,gzip,hashlib,json
SNAPSHOT_B64={'emerging-tech.json': 'H4sIAAAAAAACA8193XLbSJbmfT9FRt2MHSFKADIBJHyzQUuyJVuStaKqXO6OjokUmCLTAgEWfiTTExvRe7W3HbFP0DsTMW8wV33n3RfpJ9lzEiAJkgABUlSxK8pVFkUSicQ5ef6+850//YGQf4M/hPyk+j+9IT99ZtSzDdt2qPfTQf6LVKWBxN+9i7KwL1IVheQy6ssgIfdRTN7LUMYiUEkKf42SMbxBBKQbp+pe+fjX8zCVQaAGMvTl9DsnUsTwlZZhseKVLA7wGsM0HSdvjo76kTqM4sGRaRxaJvWOkiQODxk3mGHMFuarVK8mgQ+aljFdbjRWPr70J/3z9P5K93hrGq7tFt+iXw/EndSXhzsYxGI8VD4sG+5ulN9ub5KkcpSQXpr1lUzKH038KMbdMQ49z7UN+MeinJnc5sV7/sdB/TpM03Aq13EiUkEuRSgGciTDlIiwT7rBIIpVOhzVXh93xzAdaricGm2uTw2ndh+CaKB8eHz6UatwkK8hFMEkUXUr4NTyYBsMx3I4NacL0P//c/F4HuTkKYr75Qf0U0luxPIFfirJF64gGUuUKvxR/7X0NMoC+ko+qj5K3Ov5r4+j0ThLZUwS+NhcGEmx3SuvwrLw+YMwLdyByNJhFC/cwIdoKMJQJuSDeMju1MP8O3rZV5WSm2hSWsYwVgm5HipQivH0u/9QPKhlZeQec5lJ6YoynsFm3EXRA4nut9I72qh3oFLmkedy0zAoiBOzaJXiuWzfescZRbk3TNuyucu93YhdvbCUtlitbvE6adK3WJYEEafTF5tErBeBCr4X0fzTX0ApvypJzrL5a59l+CRDcqHWixUzHIdR1zBWxOo2ehKwTSQdShLKbykZaPXTT2NLSbObJQ0OraOvh1/F4BDfDz8z6vAqaePW/k95x8MzzjNMz3TsNoesxa3KZXSzFA8Y2YfzQfT16fY2U0EfD9vTb2ksfFzWrpaxY1vDOFyec9NlcEjZuzrqpwduW61p0LkbOYAdhGtNf9OkZWBsBv4QNOhSqLKm/ZaF5FdV+uZflYiepCIflGg4wR2XmtRk5oqqdfuPAtaUwCkCWhb5+sDJJREfwaC0U9UnDnkFu9M9f63dsJEYj0FwtjroF9SPws9w3huV6udtoH4vJPfc1XLvcNhY12wh+C/l7KECurbnuMxwrVYK6LDKddzIUZRK0pNhMvWzLtRJ94Z0x+MAfLB892uWYeEyuOFS23U5340eNrlD67SztKmqtKmJ3tT5266HkVbi0UimcbP1+wLW7ztsUTSTcHjxUsQDEcuUfBTBj7/5yYMo/y6VEdjL//e3sYijR/V//9d6NbWowWzwIFbVtNnkTfVwVffMFrrn0aPoLkJHC45ykG7wZayOYVmV9s9x92//uImWx6PUYtbvH+QwZmKMZVHThlPgJeX9GR5ghar0iqsvfKC109cQJLgmd23TWpHd02/jQKhQ3AWSFLcfTIj8hseKStuYGLBNWXwnwnzXSre0mYUxXffIonhQQYhIIbJgDq+M4x2+gYCblFWfqLexuIe7Idex7Ct/ZlJLsnYr/WEIZr1ezGGtIGcOMx04WJ02cg6fqFzNWTYSmDK5U4FKJ/n5HuXHeuetSGRjWO1xhuvxbMOzubVHn5NhoGWBCkIAYe3I6cOTNB7DMS7y/WmlgEvCiNfAA61GpZvU7Bq8vkkmMG4qxVJf4IU/QlxfsjjvYIVj8lYFX6X/0BBjUcNzbBNscm2MJcBxmyUsRphm0Z5cG8fvUSX4obEYy7jC7lgtNJLZR9S2Xc+0D6ntYMhVqY90E310zWp9vMyCVMEdYjpJ+EMVSnIBiw1R6to4OR7FAJ8asJ+228bmgJTy6pMBb2OW09qZvhkmNyuvdyXSLIbbvgA5yuDwgSMpAp9fO3rNZ9BLBFn1WnUrkgfyahxHIN0pBBTTw/L1s3Mf1Ym57YKxPH12DF9YOio+wmtwtB4Po/VaCXIEEbPLbWcLP+8NuQY/PcS3JJgMWRFl1F56Au+CvSHHQZShksMJW/rqEzUA7QrI7ZMKK0yp0cJhNNyjBJwgh4GXaHQMwwR/sVJ3+Ua6+yLRCUcB9hyLG45rtAqSLLNakWBfe1n8KCfTRRzDmaJV60yCwwiiWh8waofR8UCLuNtiDfBWXreGWTYcHKOyxLTYC4brgGPXg3t0d6PLuaT5KGk78Vs/R/HDfRA9lfQZvAW41VfgMwyl9mlLC2lW4Q8//h7DNcjJj/8KglDGDSYTolnXYhXKeZlnN3RaMoCnn/hg+bQYxODGjMQY9XGDjAnc0m9g8FOtLY+SDLWLNpiGs2/AIwM3OUWZh9/mwSvePInh+JJP25SxFrMsDLMsLmVVeuuZm+jty3mdHgqs4XGIRZndygpaNYrzsw4hwCsPk3EUF5Ger+3gktPXygCudcKZVRcS5BefZ9hme3MehtHjWuWFqFzHvK5hu8zbkSU+WxS6tWmV2pTMLMniJ9tUrnrqN9CFzwteLuY2yVm28NqCi7zewnITY/NqC4vZHtAiiPLlXSwWjs3lE7Um14IPbskknY8WHautzKppeEdCiySqp3FIwaWzWaV+upvY1RcyJhtnH3dd6vUczMMYHEyqC1btd88DcdtGjWSODRpP+d6LfrOP5qFZKb7V0jlelk4y86ySXIwbVfXn70LFpJsEYkTeDkWalpxiiFMjrZ5fSgHshQqfQInhNRE2GF54hsw17dV6IOreAOt/8D3gGHT8CAvvsGIsOUShSuEphYMDkgi9KXA7RSHioG0RA51meCEWnSLRBDc5joIg0+f0eJbBSZ5R2pDhI3xlUd2wTE6tSr3epLphmF613e3C+v97JmZG90yKIB3CITUWfrom24R/KDMcw3G8Nk6qaTVf/3L2hPRS3sH1fJGkayLfjbNeEI7Y1etIR/DshxLPe4jiRhCgxfmOCBlHSRTUajbXZ5vlWJ7NXHN/WSYIKgeLCgt7B28scuelbz8NH1UchXh6bVBsvPjxdzCnC2b2g/CjO3IiQiWDhXMpCCS40deH5K1QgZw0FDO4Da606TjPKGagTUJve3oAgL94NunHM1uh5SnIHvHz08LB1H68AQuNbjI65cdZHOOhvmDq8cO38HI/2SZ3RalnHD2ZzGTWPFrcPouM1c5qD7q431xqP8MexMlQLqSRG0olm3vQXs1a3gVRBI6PSh7AAiRgSWaWcr6a2lVoWJjnQGDlGW3OFpN563dES0DpNCE/a0ese167BKwZcViByy3WPvit19otzfhUGl/dB5nqv17yr0cqXND3hdvVCdqgjaE+FnEQJeR9FH4XgUz+8Ze/nod+uTapHmRAjgXodHkBYTghx3E0GDZYa/Cuqctd+M8qpKCkeDfSxzu9kQmoFhzASeE63wcyt6ckCrfC87RwpqllUDC7T8NsoBL8iGF7bpWiWmz/9UxDyyaF0BFUxNmDKw3ulzZ4NmWm67XJUJUyB0tGN8eU9MnxHE2C2cZZpjlfT3OAYTjaveaIPoRD9iXrrO/A1vXLGabk9dpKar12v1yNlWoZ8Zi7qnO4HBGrJAfI9WRw3/kUD8CCf8fHfynGB2XVupI6Z3klU9i6h9xJPo46XV+nofC3Uedd9v07JiXuZawtci7H5FLC2vs6Tu6pURbkTvn7GHPrT2icpm7fm63U2myZejYxpO3A++GPB3+rdKVtZ5MU1u7NDdZvXMehzHO9NtGxZc/KX0sqXdreRS+gMT5GT9qA74UAlTrt0s7Vx8rn8sMt+/RarfUG1Z5uXEfphmOCW70JJLskNmEusmEusi1NcGnfluzpQB+SJVPe/S1T9+W3XUoIA5PfMgGxH5FxHMWNSvyLGKo+eT+MIDguhcWX0VCMRgLN3ndBPg7BHA+Gpd/3hIJLXKsYHfthk9XlHLwXYxWKfSLlmNyLBznP5f038hkLV6Xwt49nH2gebAs+zZ2qJTeOTBsRTIaDMa55aHrluGyxUcLexORatMbkTutYlxCeC62bYQLm90Smcj2CQJs62EvDBt1o44vOcS1Lpi4EyxBM5pfcwsp54KjAPwgpgnW1Cncp5+stbw/iJREWcjA/M0YifmhV9NX4Rs+1HQ6OBvN2hqxtl1E+jkAav4GDjD6jyFX1dYuU9bWMfTnWT75JV99GCKsoAdl78NN3CXuzhLY4Hmbht0ySX7NG/JPlsVJ/wkwzZ9CW61g9Cn/SuY7BHY4fp45ZT/pZLNuknVd7n179kmMwrmswGLQtBoN74C8fUseynepo1nL+CZxknueGHDgBHb5XNBTnWmkxTWS6ZjsEcE1tu0IsyjYVXS1Uk7qFWLbut3I44lN2VBTauofpbZDJMbgFaXtcI0o/RtWNVVyFqa0BuVnQ2pWGlAW8RoPGOhhsUeM5CSqtwLO0s06zxxPyLtNaiVnlXqQCcj2E/UX/8TqOQFFB4qagwO3A8jr5lGRgljwTQqNKG0s360V0qxVFLx/3A784wRioyPeUgf6rhSFdqISjhBrg/LVRDj5DYlZcH1O103op6KbSaad0Wkut9X2pDh4dh3G7lVV1zOrSUA/MZkq6g1j5Gu6R1//qHX8T3V04Tm3HcGlrheypIF3rfpYcxqm8qVzemnLAdcWekhrBLqPJhZtrVMR3cPmRHILYwxFBzqIkkSos+7sTOD0G5K3A4mhZV+UE7iV3km/Ed/GoOrfgkjS4vKCm3PAst6L78EVagZuVD8TJNo5E/E09HloUTSjIWGWnirkRUP4Fsq6uro9aLhp17rp7w+y6tsYyug5i5qnXqqJDnfUu7mISo42jzXQC2GU2BWNpt6/W1pu7tUa0JwdaDxf2JUehYtNwiqvXCrwpsrHIEO+tH5gZiK9w57iEmUbmJ+UwSuFgTkl/FhRlOlXSXId9Q7pkVAAagymgEUxNHMGLJI0InIYZlmu1SsCKH54Bgkr8ZAqCsrlLn29DX6ZssmESaed5LJfq7I0D0TK1W/XavFwti3GNwWY6pWXtKCF8Vojrq8FKpFmR+M0f5miSDp+D2GjMB2tbeXlIzg5J7xAzR3DqlWxsN5DfYA8xbIzCH/8pgscyBcGliJUgH6If/x6RdyLw8f/gXjyKpoqtaXGIYe1tHGI0vaciXsiFRPda0ArShAtdrVWx3KreanLbsI4ekuTw0VUetqwx99k2l1KXrkEDqXAao2/RMAB+p6eFlTtwlfYplPXiWC/GF+KJSIyp/cUD5Sfc8ns1GC4gLip7mtEGwLeH2sIkbVoiu8FEgIjFE4SxkmsEzwzFOAH5K6EeYvDIyFtEDaSRiLOZ6aozLyZDyIXLK+gm4rBoW9ggRNOSueAXn44hpB0pvb1ocq5EHOcQ3JsaZK3dFhJv2C42TtodePCe26mE7pmbQHwY9apFNIf0HJRv7YAcR4NQ6TvpDmrjM8oMLZkUE7QOc1u1luwQaGRZPOcNMBxwxkzG94U0MpmlC4xIWuNAvLYjQPyNHGMqJ8xh3SixZAjOqg6fS6Hat3GURCO59IoOL2deQl1kJ0sSvOQd+lnOlNGou6BNCo3L+aRc9+hJNVDko4hFIh4W6AcaCT0oxNmu53irkVoXbMEj5rZyVADsXq9BedNyBgw12McHPA7Ao50+3r58lEE0Lm3TholPwztSvvT91HYc25jSDmD123Mq9Zbtu5XFzFFpnsst02CtoOgv2J1qavyvZznw2I1WHXEvE10anq5WUINCVOK+eIdadUKzTTv2ssHdEHt7Ju4TQT5l/jAuAXm6d304LAQWGN9K0I9BlMVlRzEQkpyCfVNgOBs7X1wIYhyjBl6Qm0j00CcaZbCU9pxqKgL2fpEDWTQ5TBPp5EmBYWjT4JYHJNfgYhdC2fPVg0o72vnaKuBj4NQeWZRxDiHMkfrqy86jaSprbBqVDAfU3AiEW9MttqTpKiznEGtrHNoeOY6DjHPevhg9uF6Hi6A9CD1bBX3cq4bg9nKCNHKCHqW6y2ZHzXGgUOXJMRbcZO2Jp8ncKGwyRO3tczaF9GAjqSb40NDxov319bMRfDPRb06ttu+C+UX5D7BU0j3szT/+i+oPwcW+EA/JcATmunTBQ1StsN+o0xAp29YqCvc0hH33cV9gczBjiu0BOjGlHtHWFhD6XG3bpW4Gs+qepnhYxGx0niQGIXDEo1VP8j62QYz+PApEkeTZVVMbc56N9LPM+iLmIpZ8CrlrKotsjq+vs5eXxSMrNXLrhYDv21+zACtXJoQYeqbVqmi5y7qQa+tOXAbxMIXdba/MVYLZNixeFbYqniAtsJ+nEnqz5r2TSpu/hE9qRN1n0W9qqePtg5Lky8IreHTLkHzOmvKxhu0hMUE99OBE6AagUvoRrdG9hoSAeoM1h+M5byw4hZAlTcjdhFyAgx2LQRFuC6Uj6gbr/fpNbQzdrMgUGxuPnr5COJ5rMjrkh1hof3YYjZ101YIsVTKaJyVP4Xgb/pYhjmptKpK5LkLrwPiCebTAXLfxfnlNEF1lqfE/Pyey9uqOhh9YzIMjgrP9+LwzxJUMQUqkjJsyTKel9zWpyK1QTxHmP7uDO/G1dNUrOHpJ9w7M6kR2rrAYrO6ichYLLtHPyAcxAq/j9HtjfdGihmsb3irpVi9neSXgOWcSgerNVnC79jCNaDUghurA+3XmiHYq7RffP/Odpavqhm1zBzyL/THfWZaGvWBp0bD2jAWiRp7OYhA7zQF8OwfLVcHQ29NQrij370z2alHbcc2KdpG5TQIzFEYqkfPWyw54haEEbZMiKaqIItTcYVNGsRbNnbvwLWX4CFJS+JeWRZ1KkJy5EeLc8aoV9Ervwen9Pbg8Sds64abkRC9zPDi2o7EvLrU8sx3XilPTkA1nAiKi48zXkJtLgc80FNP2+3muty5yNJyquuVzuU4KwSyZo0WJXf7FK4TPRa9bQ3Q2YE2oxdOUckNRMBjCGTZCtA7Czv/xl79eKREvpHaRauEO3LsvjRx/HjeZyc0Kfsrr2wOIr+OBnEXZea9aQl5dXFwmr4tG7JzG+VGu4UbBkk3xUbVgcoutekPErrhQELLu2prefOpv2hS0u1K1TXuTqo1VY2zeqoFGcy5CPNshxblrYLmC27BEhxvu798lVridEJZybnLakpCwBlWHj1MLQdEnNl3CEhNDzUrA5caTxnJdcElsvkuafy2gOmsk4srfafHclJN2IzLAKkaUW02IUsmQ0lBPtWzLsy32HJzreUI+Iw54yn80bb04UXi+bVcwBfWzmIOAAyunXId1utSs9Hy9/aPPq3kn9+NxOhrJZheWthUsyPXWgFuPVTo9h1r0c9k0T1Kb3KaGtaOq6VSi+mWJeiFe3JbjDtpqmIWNRJZj8Ipe6JZsQjPqks6dFgCJeQgS3WFHQC6l/qy0n1s/tI6xTMagHxKhcceffjk/6ZjeVuCa3MsF106gyxWhOlqHhn0IAlZpDDciyOWmVwuy6ZzEWCPH9BO4+DKn11NBIMqP7JnAbiwKVi5gumUgc2IQRlikytaD4RxDlxotgzPboK0yqMw01l+8VMvX9jhZewzZGlvvMtO2meXsKu+ztt+qWgN7Ks0KGrMnEWuYw1pu3C4K+4S8guM3iJII9O91ixD0XgmI/7qpLMFXjwuYUkSuZQKXTtQC9M2PyGkGF2usdlLGKHVWI9KfUziXv7dDrKLmIfOfjwIbgyAl2IedH6Z+HGnoh0bExHoKRLKdo0o5sm3aHu/A+zs2eGGsM6lurNzETtp2tWv2PoB4AMkh9D2dh37eAVTUYmMpw3VFDg0595gDxsFp01wJy6iORI9BEGOIgedLmV08dxSb+jxdnSOyLMM1aKtyx66PKkdbSrAOcFqwPSVpC0SWL2K58YCEG5lEWYxPPggKB6UZEjgcgf16J+AMK/nIH2UYSljGySE5VdifdSeyUUWPdDfA3C7YQNVMN+8w6njPcGrfrFD/FXt1PN+rjcwo6JNnHlmWyztwtnzDEULVAEBjj+aT6xDOMBkmi8wXHJpz3tNaOs7uYI+LY3DzER3Vn+5lSaozgDkjT5NMfiQ90hNDWHe5TiDlA1wUsW7psNwecRlloXqAGJT04LhpMRsNtJxXkdbFUT/LoVVgJZ4zwmObOWmdygzl3kd3cHBbNCeFjROJzP2cietruTXcinuak+ZC+G5zSpnTNLypDclhvABTwdT4ctKpJHZO28DBj5JYPmLU4GA5y6uOGvi+Zc8yTT02hnPbtj2Dt/LfnZrO1Lxo21nkfz0ORJLg5ot6v8TMp6bBf1ELONtbW54BFzfydD0OsgKfbTfa2IMoZoUr7Sa6gzNbg6FfyccoT5a/bpd6a+D/nmfIYrgErDXFEnDSnEE/DZRICJjQNBEr7TO3sRo8iAY6FsrgMbrGakIcGVDyTGbRC65jrCRtQ4tU9I3r199jt9Gfcigu3jWcdCOVplImf96KpgVBz6NBnOQULVQP2a0O8em+093gPWtL4WCiyWqXcXtJnhZMNes6LwI5WjXT7tp1M3XajyOppuUZ7VvLjz9d3Z72blv3rWpi3ftsEf/0bunnlja3SQevRATO0ZfoIZqIMpNLH6GQ7yEqUOXw/iZCgsKzH/8RJv6wcbwoNwxmeQ0VZ7HAXjYbrkH+iIXnnzdpNuqA73awzAv+6qb3OmdYQ4f4Noryutal+Crjvsjp6O90I3s8IW8h7oFz7gqPMNK9j0EiX2+TWy/ISSHsBW/QfDZ46iWIyezSsFyzVQPQjtGYbs5xVDSls3201tJ8jKRL4ZnaJt1RKFZFelY1C64orcOXYLhd0uyc6i2exv6j1VJcQ/TWPEgxQd79iegvZO5SPS0R1S/IfL/0q48igi+BNV356q4xMcCp43oVdBHXCAVH/wOFxB8i82k4kEm7oR3bGVuIDTGtPvALe6tZSJlVCWdk/wzzNpCXPHdOIZQE58baF8DLcIvWPIRKG14r/WSUueu9EJ01LOpMp34ETkHN0A+DevnlXYxXqfWiM013Usvatqnoj2fdq/cER5vCuV0uN/8RfgQtiZYJ0W4/wfuxMuYPo+bSs+F6tkXNZ7by5tleOV6ccoUY2SRQfeQHR9ByBAagOPPfkGlsfID9JYWq53b4dCRjDWo+UbGsY/VvaWPjBNm/zNIor2dk42l1lWp2m/nJFctAx3tD8R0H+O0uDe/UdAoex5MZh35Rm9ILKRUna13mfD6GZbn27zAeY7ZRzV05m/XwtaoTH4O+VOV2ysBk0fmSkbNkPZ7fgggDI6oKE9a9uT1/d3583r0g5+DSX1ycvz+9Oj4l3asT8v70U++6e4u/6151L770wN08vyIn571u7/b0hlzCq+9PL0+vblekfeYMrkltgy07Usk4TjpYJITIJel8C5Tq0M4T70Bg0sGvqeylc/YeSHqa25ozBl6W0WqgBHXsmqrtAqV1zuuWr6PoOGsT17oaxuV6nIKH7uwNQ81cqjHUHOIkau+pPNWr/SZ9TuuacZUHqul0ph8ir0odCM1lZQi+IPQ6bxzIjYTbruGa3KjoDShGKeJNLUFVS9MqWve8dotnpjGNFyrFXBh+V10zjdm2v10liU3BQOUOqGdTalX3vG6SnWWsxlaVhfAGx2zJx2JMR4lRpU4cDUurhetZnsNbYZhfkL7FsxydHLWR6Qqu1J7Hr8y1rBYko2UCBs+6fpnxswXbfEUap7QOxBQ1R2VZfyhIT6YTQa6yQRwNy+R9GRxuEQG3LYlILxuDS/gg0kbgoYujwSqGDw8lOZ0mgQtGiI2Z+lpysnhYje0Y1Oxwl1lmx/7XSnwv3TvAEOLWPBvguZ7ZakboboeZUUY3JIt/OYAjpch7BtvgYW2CvliRei2138Ylwo0Sn9WDhb+o37IQwiu5iPN9kop8UKIJ0cRc5jgVQw9PKxtmOnWFwtGcTROxvjJJVSHJoKrXl9ahTSB4xskuOdUMecSKif5bP9P0O340GuWI7jhLhmQYZbFOe2JTnwif0YgzH6KGfTigKtUAC3OjPu+XGWKmQx9qWdwzLfZ7cyxtwRy4+yFqcOMaqMmph9zou9Hi05k0llgSZvL3SuNXMbHTvuWmzSHQpM3XUnW+KPI5WhjdlMnJP/7y1w+g5L2sTDWvhvAyMssjsK2vQOUummjRKIipCQ6psRXiCaxRhscuqHlEwMgj6Hs8xArHiW6wK/lDWu1BtgJtOlDg/6Un5UORg8HdnfwLqvLPA3hFbOeZOoxjXvSr1F6pYRugzOazYf/zWXNLZAlR8lsG2tu5i2K4135ecCvywBHCdIK6SM3R2VATwyOn1XxRY95IvLQIEeiy8nQUVjF0Z+3VLW2QLZz2axntYfb6kbYdkbJq5n4BvwxkRyLz9KAyz1/uhxvrGUALhcENgDJvZfhVjHDy7CHRYwdLxlSGENhdRAswQYRqCfJR9PvZKHsQSROjpS7aGhWMRl2U9HexGEnka3hDTrCPPMmj+mngKdHcdc/xKcHJEGUJZiwPyCed95Jx5zxEMlp8ikEgxomcDlLK+QRyvnec9QxR4iSEu/S3mh7KQAqOAoXyfzhQxRGVk5Wx6so9tiFvojeGVU2NcIsfnrWH1YJsc0Q8kr0zo1VDuMmrVeRK5OWoi2kHYYndpJGMzGPa3rqI+uWtKKapY1VX528RC38jRV+U/NeegiOxAdnjcT0qHLYEc8KmswOG6VuRPGgWo6/ST0uJkNcLac9Sw2Wp/7TYzllDZtX4357ErtpdJEFv1GMEmpkNZCzCMtDyNFDgelb+CrXwgLw6DkTWl6QXIUaYsEPndYOj6xhIbO5aqzU+GRdMMbLlaEFyDecXnmy5v6vpuWEjcaAofjCCwG0cxUWmJ5Fp8lx0JnKsWbyaGXT/M1ZMT1ODghmzKcTuL50TrAjJnk2v1zgZ85D8ckg+xioZhuIBItV4mQLzSabiUZD3qDiBKpNxgbUj13AcNA2/pWC8LZfz1eaSm9Pbm+7x7ekJwWT+8cXpG7JRv8nGIE7WloDEgtBR95eYps1ZtYC6eycg4XqcrelR27N5q1z+jke+awpyC/xBahpsH0Pvua07wEwMqQy6IwT1W4Uz8Qq1qx0qXcqytx8u0ibt0swzrWKcntcbghN9X1LJM9FX8AyHfTVG3sumbD5jrlsxPO9S+XE0I5dFFevLRA30T4/C98F+5T2YAmJueF3nMnAS/QwkA5YBY6WkoMFr0OKiyfN+6n9uN3FPJ0Ty5eWB1KHhHc55rrZvOdl1z+IW9be6FZyPRhnEcNPG1zyIKz2i+lSEp2fKux63GKf76AXzdC8Yd2wTB7Lsio+kIv6qtbazdq5XEAOL0Z0a5B2cr3dHXHA7VGIQkW42yJI0ImcyDn/8LUTswk3kD8XCqGk4LN7DtVQG3iG4f91gJFVfLNj8oSIfo+9DMNKjZmoSSpnF12VH1tnUaZIzxNrLBP6eo05CZATUP4UqxbmhIemr6BuCTmpSoKU8Jx4G02uP5zzLZT99B2REmIZ/dhJ0x8lHbmigsks585i3L373LQaR4YDVxiSoRhEtGMb32EewGOSvnkC6nkgZpYiHaZ/NyaWxFNctCWLLTE9DEnRaLtRyvNy8Ub75mY/QOH9IgjmVcCw+SvJW3AlRzhtVpUuvRVakRQc5mma9xhsu445TofE/w3kTJ6nQVKi6gjGeobrLvrJO79REhYtkzTijM52NYNuKWkH4OakCvGQYlblOblubKOyuptVpfxq0w7I42OdW+Zsd8l2zhXCzfaGv6ptLLy76qiU3F/kdFpgKpmfLAi/BQYl5Fls46wbDX887PF9JjSfVv2/En0TDPvkAISZsaYmrS8HpKgJZbn89v4vFUI2K13F4dJNWaGIY03MqpuXpSTxBpBHQBW5Gp8BF4BdOxCZkzz76PcmUDH4sxpnADNOj2poGUysKJqmSQlXgX2Y/u5OQ1vhxmNadAhgPCARAfTip4gSEIU/kzrylZvY9RzuXsGhsJ2hTubfmUcjioj7htFARjEhJtHT6cbHheVes7cyowbkV0nLqz6lxy/u1GS9Rs42T9/p6RQ3PX/JgXy/EqbPHtIj8BH/scanlYdmTLW5K+svD3wdxlC+gxcijd/AMBBb04omApyRGoAElp/VCJoECg7dg2U5xVJdAHzgIVBNomnLHcGxme5Vl/Nz9OYWTdmG4QtFpUm3NDsp50IqcYIGL/gZGsq/Zr/MjEd3in+M7vFmc3eLLZOs5KxCgwambVy0oWG6HVmJFmf3P0ASxDU/6vM1qcS2/IPvjQniYk2zFwteT2C8lCFi/fgxDzrfnOrZjtzPOTk2yaSoxOXcmziOUBaNUvTabmuIayYYMe1cgnProdC6xODrSzwebJ9kYZ/yu8EK8UzLo15nlGiIUFIt//OV/z04XtGjTZGNzR/9IKCWHoMIPQ5xvBk60HKlNqTZ7UdS5VODdRk1tTCbCJlzHrOCVSDAmTfJafslWDyXcDgYGEqUebHIw649IssSX41QVGpFnn5CWAnzj3ln3uvPr+7dRBEdjTWhKtwhNKWi+S+3KIX0Os/bTGWG24yOqvhwO0iOyoOWF3QMFysWyPlObj6tk4Fw7Xqva6AvMEK3BFTaqcUU7RQvyzEB9n5ZNQbqi8nDqXpWglpzp6ZD0warlvgC/NDfajVFnFk7Ucs9S5Yu/ghkfBMjUcSmaWok9CpGmU+FU68fSW1SvxSkLqGO3Ep0beHzgoeRNvgdgh/tKgCMdktlI4xbVyu50GG5pOMMCFr2brUwd3maOoQXCks8xpCA/HevZXviLDMfdGGL3YpNpPVv3WyGogBqsHXi3hhh3cS0ncZTBcd/oshg6jHcYZ3Ds7KhuWzXrdgVWPre6MQYoA7ldO3FL0mvsAhThV+2ThuWSrYizMWhA0jTsweU4XpRVMHXeqbtARSPsWpjSSi/3BusdKqxNXTI5mDdvgPaLlTzFlrMhuJ4NYRrc9jpPlbONnH1kjzgWNVzLM22Heez3zh5xqhn+qENdm1mtBplallPrS+DAlTwUmlr25chnpUlEq93KAp7LnxONs6QqrdWa2KqBVW3z+qsm9fso4AujrOzPipGKyWlAPogf/+d+YezvmUgSUNPjoQBzFWWiETFhchtOcrNCM/MmqE1oON7MEawz4PpKExC8nIe310Xld5uaDESKGtrT8VgHwkXaoQ7rmP9Kn9/j/3KTO12Wt3q43DJMq5Xavliw7Th5jIvNHpa5Qc5oyyizDZCoHSRpA8QR+K4KOTGz742ktCZz3Co6wc204PRTpRZ8qtCCT7vQApuDtB59HanDR6pMMFvVcd8/jfznxCse0p5R19yr/GMKFuUfZzsYO4IFNJMZ1uvHUoFMNH3VVKCCQqBeVkEYcqoytwL62bLO358xgOGjmg65hBtNRiIISM5ldBdplgPwQWa87ETpIbeT7WMqzM2ZOqbCCQJO59ka8iIx1caV+xeZP1uwryz3/x38/mjDfN4Jw+PI2RU/Yo1MrcHQVQycBR81jb6p5VpG+1Cqi8gVci2yEma8F6uvCNUOv6pAlF/PxqDd8OZ0KB4aUXbUw+GIqyasWi0PCvcUoy4Q2aIakVQVvLYByOEk+STyc4AcjndnldV3MAmbqB7n1fW703Q4JYQoGBwLpEzeS1JbBdfYEDjWmOtY7Trfa9L8s8ilX8BRitRoAvso69Fplq2pBW3bYCb8ad/tXl2XRKhZcUGdpJcxbEVSVc9bKchPx0L7i2Oh1xI+lPLr9ylOI4BjfhxEE33QtZi7nIJheC8C8b1k1tQgkwH58T9hEQE51m0SURnAgnbi8yE5Bqshmsvy3GGuZVVAT9E7UHhHvbFIUF6uZZzd4TeSXjbM8EfsdIIfT3ACO/z/YwTe6Efdgk8uYRsGWSge0H/D6t2DfsvLck3bjmWaR3cZPIzDRxPcPVrdiLRJMe9FqRyYk6cJTAfC9VYjQ6hRY00+z7gOL1rZU2aYmmzdMjyXua0MmW1X4xVO+342JZ6ZseBMpjXcOr3WqDOs39kuZzuKrhqgZO+mpfRGs1bw55SSBrPc56tS5qzFoBB4GBNwL54WQGZDOIIEOR7qeSHwwNJMNqJnKIQ7ll3F5q45XVDiLqKnzjFWywqm1Dks8YDMbXvBRHYw5aRsQ/qLnPGik4djCHycjRssBTtbFeEtk3pHSRKHh4xCuOF6z+0+eqmm+U0npO+6aV5bQYeBslitsvgv0DTv6KZ5zCwiDmHHLmezTuLGVUyzbND5+WDMxtHnP/4OirpYnsPOX7BvoZKlbdFkv3JCUI0DNS5n9v/w5/8PiWUTWTTSAAA=', 'provenance.json': 'H4sIAAAAAAACA4VW708bORD9fn/FCFG+HLsJOQpqo6jKEXrHqUpRoTrdJ+SsZ3dddm3X9gZyVf/3e94fSWhBJyHwesb2zJv3Zvj2C9GB18L60oSFCHzwlg4m48lZMn6TnJweHEc71+wKpYtbzkrYv2EPu9aZtZLs4omPlvW84sfWH7bGVXG7DMH6t6ORsCo1cBFwSY0rRg/G3ft3noXLylnBxlsRlKheTcbCBZWrrPtQOnBVqYJ1xke5qgK7We5MfWebVaUynDH6TiLqt5PxyZtkfIKfI8susaLg2dl4CMdxZpz0COls3G9ZY5uqvSAG+l45H2Ck+fUVdWGRY99UwR+TNoEE+Y0PXONEBsta8cMUf4NTvBYV4XZ28VsozTLdwqBViNdf78IdTMEJ7XPj6mj/bCkYCqVjpgHWRHivCtwGk1WZJ+RFVuD3lFbKRpwCdxtJ60EsC/YkHFNWxcN5/2SfQqZC94loac0Odly+/HBNyMJW7LdhI1bWnvcLSzVyA9KClKeLi3FKN4h1qK80mX9aYCGlio+JKim5sqNcfD3A5d9bPpV4PKBCz3Np+c8FfRBa1gIcoWuUgd26jZwuTF0rZIbliKJfDI9AW/EC8WLEKRLfmFzzwwasSxs/+tM0HnROFrzmytiadRgtOKINKskEbycXyERLYCmT3xtVSbj71nAD1P1IZ46Tr+WjH9792rDb/P/LSMY0LuPdBekXb/S7wwdgwjOPCjV+9mpyvri8ufpjOb+9XOADSpgvsRhX9V3YWI4OV1oqQNa0Ohngwv7RYaVqFWavx+Px0WHLy1ll73RTr9gdr5Tul0eHnivOwp4Rt2tR8/HKONMUpZLHkn1U13EUSmgkFkYX3Wp30wCB0rYJA1iftoqbjM9ebz2gFSUXzljLErbz058EenL6+vwH9exSTao+UZJ9uSIVrhbPiup92y9ohyT9SrurthRLacGysa0+mbZoTClve0IbMa36tCgziFNFnmzVPu1XW6c+d/g2OsTrd6FGNKOCWj1CisE12c4AXWZgeMx7Ex9u+DlJds3kKf238pxSYaDt2DZWmx98btnVEH9Onz0PXW0FRO85REnHrqHqlJaGLIaBKZywZczCqr2WtkJS0EXL9bmslVZIAsmteb8mHpjkkV6kkCQY0/UC6jiHdfd+KJnypqoIKSNqzzG6DF23cXAe2kTa5mFW2z7gSZr2PEMwAMOX9JfSj2oPPc0s0dC2LQctPbwwu5bzmzldflxe3tL6t58nxuR88gMf45iUhN6hA2F2AXq36fKBBxpBb0I1LRs0VhJ5jqzB+OebFBvNIS18nqVaeJGigm1HHe3CeULsT9wAKXg0sXMhlKwyHo2qezbSuDBuQ8NU9xRHZov138z3dBpD5ZUx99OeBIANYYKApfhXONwHqjX+2ZLftO0LJ1AulIYgoj5dyTnI0JJ5DecpCRQMc3uguxc5ozDxgDUqBhrlMVhVbUUW9gqG/xAq+ULBRMePy1hTWitBrXMiAsbeCxija5XNKkWmIwxLE5vu6JlTezJ7+kr3DwdAr6H0KXnEW3UDNIO908qG8lbBqMlj6AgxQEdCfmlkP4+7LH/5/h/FnwKifAkAAA=='}
SNAPSHOT_HASHES={'emerging-tech.json': 'aec84855a8d67fb4094b4953327a8ebb2209055eaabd84c5313130813ce748b0', 'provenance.json': 'b3b3c4360e247caf8c3035a8bf926d03b115ecd47c04ce127037dec02291f4b1'}
def bundled_json(name):
    raw=gzip.decompress(base64.b64decode(SNAPSHOT_B64[name]))
    assert hashlib.sha256(raw).hexdigest()==SNAPSHOT_HASHES[name]
    return json.loads(raw)
import matplotlib.pyplot as plt
from pathlib import Path
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':11,'axes.titlesize':14,'axes.labelsize':11,'xtick.labelsize':10,'ytick.labelsize':10,'figure.dpi':120,'axes.spines.top':False,'axes.spines.right':False,'svg.fonttype':'none'})
LESSON_OUTPUT=Path('lesson_outputs');LESSON_OUTPUT.mkdir(exist_ok=True)
def save_lesson_figure(fig,name):
    for fmt in ('svg','pdf','png'):fig.savefig(LESSON_OUTPUT/(name+'.'+fmt),bbox_inches='tight',dpi=160)
    plt.show()
import networkx as nx,pandas as pd,numpy as np,textwrap
NETWORK_PARAMETERS={'maxPapers':30,'since':2019,'seed':301,'metric':'degree'}
papers=[p for p in bundled_json('emerging-tech.json') if p['year']>=NETWORK_PARAMETERS['since']][:NETWORK_PARAMETERS['maxPapers']]
paper_topic=nx.Graph()
for p in papers:
    paper_topic.add_node(p['id'],kind='paper',label=p['title'])
    for t in p['topics']:
        paper_topic.add_node(t['id'],kind='topic',label=t['label']);paper_topic.add_edge(p['id'],t['id'],relation='classified under',source=p['url'])
paper_ids=[p['id'] for p in papers]
topic_ids=sorted([n for n,d in paper_topic.nodes(data=True) if d['kind']=='topic'],key=lambda n:(-paper_topic.degree(n),n))
print(len(papers),'papers;',len(topic_ids),'topics;',paper_topic.number_of_edges(),'provider-assigned links')
assert all(paper_topic.nodes[a]['kind']!=paper_topic.nodes[b]['kind'] for a,b in paper_topic.edges())
assert sum(dict(paper_topic.degree()).values())==2*paper_topic.number_of_edges()
paper_dictionary=pd.DataFrame([{'paper_label':f'P{i+1:02d}','id':p['id'],'year':p['year'],'title':p['title'],'url':p['url']} for i,p in enumerate(papers)])
topic_dictionary=pd.DataFrame([{'topic_label':f'T{i+1:02d}','id':n,'topic':paper_topic.nodes[n]['label'],'assignments':paper_topic.degree(n)} for i,n in enumerate(topic_ids)])
display(HTML('<details><summary>Inspect the paper dictionary</summary>'+paper_dictionary.to_html(index=False,escape=True)+'</details>'))
paper_dictionary.to_csv(LESSON_OUTPUT/'paper_dictionary.csv',index=False);topic_dictionary.to_csv(LESSON_OUTPUT/'topic_dictionary.csv',index=False)
fig,ax=plt.subplots(figsize=(11,5.6),layout='constrained');top=topic_ids[:8][::-1]
ax.barh([textwrap.fill(paper_topic.nodes[n]['label'],38) for n in top],[paper_topic.degree(n) for n in top],color='#14877D')
ax.set(title='Top 8 topic-assignment counts in the fixed sample',xlabel='Paper–topic links (count)');save_lesson_figure(fig,'network_topic_counts')
fig,ax=plt.subplots(figsize=(9,11),layout='constrained')
pos={n:(0,1-i/max(1,len(paper_ids)-1)) for i,n in enumerate(paper_ids)};pos.update({n:(1,1-i/max(1,len(topic_ids)-1)) for i,n in enumerate(topic_ids)})
nx.draw_networkx_edges(paper_topic,pos,ax=ax,edge_color='#9BAEB5',alpha=.55,width=.8)
for kind,color,shape in [('paper','#14877D','o'),('topic','#D97706','s')]:
    nodes=[n for n,d in paper_topic.nodes(data=True) if d['kind']==kind];nx.draw_networkx_nodes(paper_topic,pos,nodelist=nodes,node_color=color,node_shape=shape,node_size=95,ax=ax,label=kind.title())
for i,n in enumerate(paper_ids):ax.text(-.03,pos[n][1],f'P{i+1:02d}',ha='right',va='center',fontsize=10)
for i,n in enumerate(topic_ids):ax.text(1.03,pos[n][1],f'T{i+1:02d}',ha='left',va='center',fontsize=10)
ax.set_title('Ordered bipartite node-link diagram · the same assignments');ax.set_xlim(-.12,1.12);ax.set_ylim(-.06,1.07)
ax.text(.5,-.055,'P/T IDs match the downloadable source dictionaries',ha='center',fontsize=10);ax.legend(frameon=False);ax.axis('off');save_lesson_figure(fig,'network_paper_topic')
adjacency=pd.DataFrame([[int(paper_topic.has_edge(p,t)) for t in topic_ids] for p in paper_ids],index=paper_ids,columns=topic_ids);adjacency.to_csv(LESSON_OUTPUT/'paper_topic_full_matrix.csv')
shown=topic_ids[:12];fig,ax=plt.subplots(figsize=(12,9),layout='constrained')
ax.imshow(adjacency[shown].values,aspect='auto',cmap=__import__('matplotlib').colors.ListedColormap(['#F3F6F7','#14877D']),vmin=0,vmax=1)
ax.set_yticks(range(len(paper_ids)),[f'P{i+1:02d}' for i in range(len(paper_ids))]);ax.set_xticks(range(len(shown)),[textwrap.fill(paper_topic.nodes[n]['label'],20) for n in shown],rotation=45,ha='right')
ax.set(title=f'Bipartite matrix · {len(shown)} of {len(topic_ids)} topics shown',ylabel='Paper ID (same source-table order)',xlabel='Topics ranked by assignment count · teal = assigned');save_lesson_figure(fig,'network_bipartite_matrix')
print('Topic assignment is not citation. NetworkX and D3 layouts differ numerically; IDs and edges agree, coordinates need not.')

## Part A  One network, four layouts

Zachary's Karate Club is an observed social network of 34 club members and 78 ties. The split attribute lets us compare visual patterns with a known social outcome.

### Karate Club source table

| Decision / 决策 | This cell / 本单元 |
|---|---|
| Idiom / 视觉表达 | Karate Club source table |
| Task / 任务 | Inspect, compare, or explain the stated evidence. |
| Algorithm / 算法 | Inspect the named implementation and parameters in the following cell. |
| Data and units / 数据与单位 | Use the adjacent source note, units, sample, exclusions, and baseline. |
| Evidence boundary / 证据边界 | A display does not establish causality, consent, or community value. |

**Try and explain / 尝试并解释:** keep the baseline, change one documented parameter, and explain which reading changes. Retain the original and revised outputs.

In [ ]:
G = nx.karate_club_graph()
print({"nodes": G.number_of_nodes(), "edges": G.number_of_edges(), "density": round(nx.density(G), 3)})
pd.DataFrame(G.edges(), columns=["source", "target"]).head()

### Node-link layout small multiples

| Decision / 决策 | This cell / 本单元 |
|---|---|
| Idiom / 视觉表达 | Node-link layout small multiples |
| Task / 任务 | Inspect, compare, or explain the stated evidence. |
| Algorithm / 算法 | Inspect the named implementation and parameters in the following cell. |
| Data and units / 数据与单位 | Use the adjacent source note, units, sample, exclusions, and baseline. |
| Evidence boundary / 证据边界 | A display does not establish causality, consent, or community value. |

**Try and explain / 尝试并解释:** keep the baseline, change one documented parameter, and explain which reading changes. Retain the original and revised outputs.

In [ ]:
layouts = {
    "Spring: topology-first": nx.spring_layout(G, seed=301),
    "Kamada-Kawai: path-distance": nx.kamada_kawai_layout(G),
    "Spectral: eigenvectors": nx.spectral_layout(G),
    "Circular: order-first": nx.circular_layout(G),
}
club_color = [TEAL if G.nodes[n]["club"] == "Mr. Hi" else PURPLE for n in G]
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for ax, (label, pos) in zip(axes.flat, layouts.items()):
    nx.draw_networkx(G, pos, ax=ax, node_color=club_color, node_size=190,
                     edge_color="#B9C4C8", width=.8, font_size=7)
    ax.set_title(label, loc="left", fontweight="bold", color=NAVY)
    ax.axis("off")
fig.suptitle("Layout changes geometry, not the underlying ties", fontsize=16, fontweight="bold", color=NAVY)
fig.tight_layout()
fig.savefig(OUTPUT / "network_layout_comparison.png", bbox_inches="tight")
plt.show()

**Interpretation check.** A layout is an algorithmic hypothesis about useful geometry. Spring and Kamada-Kawai layouts can reveal topology, but distance on the page is not geographic distance or causal influence. Circular layouts preserve a chosen order and make crossings easier to compare, but do not reveal clusters automatically.

## Part B  Measures and communities

Use measures only when they answer a stated task. Degree supports local-connectivity questions, betweenness supports possible brokerage, PageRank supports recursive prominence, and k-core supports cohesive-subgraph questions.

### Node metric table

| Decision / 决策 | This cell / 本单元 |
|---|---|
| Idiom / 视觉表达 | Node metric table |
| Task / 任务 | Inspect, compare, or explain the stated evidence. |
| Algorithm / 算法 | Inspect the named implementation and parameters in the following cell. |
| Data and units / 数据与单位 | Use the adjacent source note, units, sample, exclusions, and baseline. |
| Evidence boundary / 证据边界 | A display does not establish causality, consent, or community value. |

**Try and explain / 尝试并解释:** keep the baseline, change one documented parameter, and explain which reading changes. Retain the original and revised outputs.

In [ ]:
metrics = pd.DataFrame({
    "node": list(G.nodes()),
    "club": [G.nodes[n]["club"] for n in G],
    "degree": pd.Series(dict(G.degree())),
    "degree_centrality": pd.Series(nx.degree_centrality(G)),
    "betweenness": pd.Series(nx.betweenness_centrality(G)),
    "pagerank": pd.Series(nx.pagerank(G)),
    "clustering": pd.Series(nx.clustering(G)),
    "k_core": pd.Series(nx.core_number(G)),
}).sort_values("betweenness", ascending=False)
metrics.head(10).style.format({"degree_centrality":"{:.3f}", "betweenness":"{:.3f}", "pagerank":"{:.3f}", "clustering":"{:.3f}"})

### Community-colored node-link diagram

| Decision / 决策 | This cell / 本单元 |
|---|---|
| Idiom / 视觉表达 | Community-colored node-link diagram |
| Task / 任务 | Inspect, compare, or explain the stated evidence. |
| Algorithm / 算法 | Inspect the named implementation and parameters in the following cell. |
| Data and units / 数据与单位 | Use the adjacent source note, units, sample, exclusions, and baseline. |
| Evidence boundary / 证据边界 | A display does not establish causality, consent, or community value. |

**Try and explain / 尝试并解释:** keep the baseline, change one documented parameter, and explain which reading changes. Retain the original and revised outputs.

In [ ]:
louvain = list(nx.community.louvain_communities(G, seed=301))
label_prop = list(nx.community.asyn_lpa_communities(G, seed=301))

# Specialist Leiden comparison is optional; the core plot uses Louvain.
leiden=louvain
if RUN_PACKAGE_EXTENSIONS:
    import igraph as ig
    import leidenalg
    edge_list = list(G.edges())
    ig_graph = ig.Graph(n=G.number_of_nodes(), edges=edge_list, directed=False)
    leiden_partition = leidenalg.find_partition(ig_graph, leidenalg.ModularityVertexPartition, seed=301)
    leiden = [set(c) for c in leiden_partition]

summary = pd.DataFrame([
    ["Louvain", len(louvain), nx.community.modularity(G, louvain)],
    ["Label propagation", len(label_prop), nx.community.modularity(G, label_prop)],
    
], columns=["algorithm", "communities", "modularity"])
display(summary.style.format({"modularity":"{:.3f}"}))

membership = {node: i for i, group in enumerate(leiden) for node in group}
pos = nx.spring_layout(G, seed=301)
palette = [TEAL, PURPLE, GOLD, CORAL, NAVY]
fig, ax = plt.subplots(figsize=(9, 7))
nx.draw_networkx_edges(G, pos, ax=ax, edge_color="#C8D0D3", width=.9)
nx.draw_networkx_nodes(G, pos, ax=ax,
                       node_color=[palette[membership[n] % len(palette)] for n in G],
                       node_size=[140 + 2200 * metrics.set_index("node").loc[n, "pagerank"] for n in G],
                       edgecolors="white", linewidths=1)
nx.draw_networkx_labels(G, pos, ax=ax, font_size=8)
ax.set_title(("Leiden" if RUN_PACKAGE_EXTENSIONS else "Louvain") + " communities; node area encodes PageRank", loc="left", fontsize=15, fontweight="bold", color=NAVY)
ax.text(0, -0.06, "Community output is algorithm- and parameter-dependent; inspect stability before interpreting.",
        transform=ax.transAxes, fontsize=9, color="#4F646B")
ax.axis("off")
fig.tight_layout()
fig.savefig(OUTPUT / "network_community_centrality.png", bbox_inches="tight")
plt.show()

## Part C  Interactive, hierarchical, and circular idioms

### Interactive node-link inspection

| Decision / 决策 | This cell / 本单元 |
|---|---|
| Idiom / 视觉表达 | Interactive node-link inspection |
| Task / 任务 | Inspect, compare, or explain the stated evidence. |
| Algorithm / 算法 | Inspect the named implementation and parameters in the following cell. |
| Data and units / 数据与单位 | Use the adjacent source note, units, sample, exclusions, and baseline. |
| Evidence boundary / 证据边界 | A display does not establish causality, consent, or community value. |

**Try and explain / 尝试并解释:** keep the baseline, change one documented parameter, and explain which reading changes. Retain the original and revised outputs.

In [ ]:
# Plotly: inspect node details without labeling every mark.
edge_x, edge_y = [], []
for u, v in G.edges():
    edge_x += [pos[u][0], pos[v][0], None]
    edge_y += [pos[u][1], pos[v][1], None]
edge_trace = go.Scatter(x=edge_x, y=edge_y, mode="lines", hoverinfo="none", line=dict(width=.7, color="#BBC6C9"))
ordered = list(G.nodes())
node_trace = go.Scatter(
    x=[pos[n][0] for n in ordered], y=[pos[n][1] for n in ordered], mode="markers",
    text=[f"Node {n}<br>Club: {G.nodes[n]['club']}<br>PageRank: {nx.pagerank(G)[n]:.3f}" for n in ordered],
    hoverinfo="text", marker=dict(size=[10 + 90 * nx.pagerank(G)[n] for n in ordered],
                                   color=[membership[n] for n in ordered], colorscale="Viridis", line=dict(width=1, color="white")))
fig = go.Figure([edge_trace, node_trace])
fig.update_layout(title="Interactive inspection: Karate Club", showlegend=False, margin=dict(l=10,r=10,t=45,b=10),
                  xaxis=dict(visible=False), yaxis=dict(visible=False), height=600)
fig.write_html(OUTPUT / "karate_plotly.html", include_plotlyjs="cdn")
fig.show()

### Physics-enabled node-link diagram

| Decision / 决策 | This cell / 本单元 |
|---|---|
| Idiom / 视觉表达 | Physics-enabled node-link diagram |
| Task / 任务 | Inspect, compare, or explain the stated evidence. |
| Algorithm / 算法 | Inspect the named implementation and parameters in the following cell. |
| Data and units / 数据与单位 | Use the adjacent source note, units, sample, exclusions, and baseline. |
| Evidence boundary / 证据边界 | A display does not establish causality, consent, or community value. |

**Try and explain / 尝试并解释:** keep the baseline, change one documented parameter, and explain which reading changes. Retain the original and revised outputs.

In [ ]:
# Optional package extension: Physics-enabled node-link diagram
if RUN_PACKAGE_EXTENSIONS:
    try:
        # PyVis: physics, drag, hover, and neighborhood inspection.
        from pyvis.network import Network
        pyvis_net = Network(height="560px", width="100%", bgcolor="#FFFFFF", font_color=NAVY, notebook=True, cdn_resources="in_line")
        for n in G:
            pyvis_net.add_node(n, label=str(n), title=f"Club: {G.nodes[n]['club']}", value=G.degree(n), group=membership[n])
        for u, v in G.edges():
            pyvis_net.add_edge(u, v)
        pyvis_net.toggle_physics(True)
        pyvis_path = OUTPUT / "karate_pyvis.html"
        pyvis_net.write_html(str(pyvis_path), open_browser=False)
        display(HTML(pyvis_path.read_text(encoding="utf-8")))
        _extension_status.setdefault('Physics-enabled node-link diagram','completed')
    except Exception as exc:
        _extension_status['Physics-enabled node-link diagram']='unavailable: '+type(exc).__name__
        print('Extension unavailable:',str(exc)[:240])
else:
    _extension_status['Physics-enabled node-link diagram']='not run (optional)'
    print('Physics-enabled node-link diagram','not run; enable extensions and install its package to explore.')

### Directed process diagram

| Decision / 决策 | This cell / 本单元 |
|---|---|
| Idiom / 视觉表达 | Directed process diagram |
| Task / 任务 | Inspect, compare, or explain the stated evidence. |
| Algorithm / 算法 | Inspect the named implementation and parameters in the following cell. |
| Data and units / 数据与单位 | Use the adjacent source note, units, sample, exclusions, and baseline. |
| Evidence boundary / 证据边界 | A display does not establish causality, consent, or community value. |

**Try and explain / 尝试并解释:** keep the baseline, change one documented parameter, and explain which reading changes. Retain the original and revised outputs.

In [ ]:
# Optional package extension: Directed process diagram
if RUN_PACKAGE_EXTENSIONS:
    try:
        # Graphviz: use a directed, hierarchical idiom for an explanation path.
        from graphviz import Digraph
        dot = Digraph("network_reasoning", format="png")
        dot.attr(rankdir="LR", bgcolor="transparent")
        for node, color in [("task", TEAL), ("model", PURPLE), ("layout", GOLD), ("evidence", CORAL), ("claim", NAVY)]:
            dot.node(node, node.title(), shape="box", style="rounded,filled", fillcolor=color, fontcolor="white")
        dot.edges([("task", "model"), ("model", "layout"), ("layout", "evidence"), ("evidence", "claim")])
        try:
            graphviz_path = dot.render(filename="network_reasoning_graphviz", directory=str(OUTPUT), cleanup=True)
            display(HTML(f"<p><b>Graphviz output:</b> {graphviz_path}</p>"))
        except Exception as exc:
            _extension_status['Directed process diagram']='fallback (not fully completed)'
            source_path = OUTPUT / "network_reasoning_graphviz.gv"
            source_path.write_text(dot.source, encoding="utf-8")
            display(HTML(f"<p><b>Graphviz source saved:</b> {source_path}. The local renderer was unavailable ({type(exc).__name__}); Colab normally installs the Graphviz executable.</p>"))
        _extension_status.setdefault('Directed process diagram','completed')
    except Exception as exc:
        _extension_status['Directed process diagram']='unavailable: '+type(exc).__name__
        print('Extension unavailable:',str(exc)[:240])
else:
    _extension_status['Directed process diagram']='not run (optional)'
    print('Directed process diagram','not run; enable extensions and install its package to explore.')

### Inter-community chord diagram

| Decision / 决策 | This cell / 本单元 |
|---|---|
| Idiom / 视觉表达 | Inter-community chord diagram |
| Task / 任务 | Inspect, compare, or explain the stated evidence. |
| Algorithm / 算法 | Inspect the named implementation and parameters in the following cell. |
| Data and units / 数据与单位 | Use the adjacent source note, units, sample, exclusions, and baseline. |
| Evidence boundary / 证据边界 | A display does not establish causality, consent, or community value. |

**Try and explain / 尝试并解释:** keep the baseline, change one documented parameter, and explain which reading changes. Retain the original and revised outputs.

In [ ]:
# Optional package extension: Inter-community chord diagram
if RUN_PACKAGE_EXTENSIONS:
    try:
        # pyCirclize: chord-like view of ties between detected communities.
        from pycirclize import Circos
        comm_ids = sorted(set(membership.values()))
        matrix = pd.DataFrame(0, index=[f"C{i+1}" for i in comm_ids], columns=[f"C{i+1}" for i in comm_ids])
        for u, v in G.edges():
            a, b = f"C{membership[u]+1}", f"C{membership[v]+1}"
            matrix.loc[a, b] += 1
            if a != b:
                matrix.loc[b, a] += 1
        circos = Circos.chord_diagram(matrix, cmap="tab10", label_kws=dict(size=11))
        fig = circos.plotfig()
        fig.suptitle("Circular relationship idiom: ties among communities", fontsize=14, fontweight="bold", color=NAVY)
        fig.savefig(OUTPUT / "network_pycirclize.png", bbox_inches="tight")
        plt.show()
        _extension_status.setdefault('Inter-community chord diagram','completed')
    except Exception as exc:
        _extension_status['Inter-community chord diagram']='unavailable: '+type(exc).__name__
        print('Extension unavailable:',str(exc)[:240])
else:
    _extension_status['Inter-community chord diagram']='not run (optional)'
    print('Inter-community chord diagram','not run; enable extensions and install its package to explore.')

## Part D  Scale-up view with HoloViews and Datashader

### Rasterized node-link exploration

| Decision / 决策 | This cell / 本单元 |
|---|---|
| Idiom / 视觉表达 | Rasterized node-link exploration |
| Task / 任务 | Inspect, compare, or explain the stated evidence. |
| Algorithm / 算法 | Inspect the named implementation and parameters in the following cell. |
| Data and units / 数据与单位 | Use the adjacent source note, units, sample, exclusions, and baseline. |
| Evidence boundary / 证据边界 | A display does not establish causality, consent, or community value. |

**Try and explain / 尝试并解释:** keep the baseline, change one documented parameter, and explain which reading changes. Retain the original and revised outputs.

In [ ]:
# Optional package extension: Rasterized node-link exploration
if RUN_PACKAGE_EXTENSIONS:
    try:
        import holoviews as hv
        from holoviews.operation.datashader import datashade, bundle_graph
        hv.extension("bokeh")
        hv_graph = hv.Graph.from_networkx(G, nx.spring_layout, seed=301).opts(
            node_size=8, node_color="index", cmap="Viridis", edge_color="#8EA1A7", width=720, height=480,
            tools=["hover"], title="HoloViews graph; Datashader route for larger networks"
        )
        display(hv_graph)
        _extension_status.setdefault('Rasterized node-link exploration','completed')
    except Exception as exc:
        _extension_status['Rasterized node-link exploration']='unavailable: '+type(exc).__name__
        print('Extension unavailable:',str(exc)[:240])
else:
    _extension_status['Rasterized node-link exploration']='not run (optional)'
    print('Rasterized node-link exploration','not run; enable extensions and install its package to explore.')

## Part E  Knowledge graph and NLP with OpenAlex

The snapshot is real scholarly metadata retrieved from OpenAlex. We build a heterogeneous graph of papers, authors, and topics, then add paper-to-paper similarity edges from TF-IDF. Similarity is a text-model output, not proof of intellectual influence.

### OpenAlex source table

| Decision / 决策 | This cell / 本单元 |
|---|---|
| Idiom / 视觉表达 | OpenAlex source table |
| Task / 任务 | Inspect, compare, or explain the stated evidence. |
| Algorithm / 算法 | Inspect the named implementation and parameters in the following cell. |
| Data and units / 数据与单位 | Use the adjacent source note, units, sample, exclusions, and baseline. |
| Evidence boundary / 证据边界 | A display does not establish causality, consent, or community value. |

**Try and explain / 尝试并解释:** keep the baseline, change one documented parameter, and explain which reading changes. Retain the original and revised outputs.

In [ ]:
OPENALEX_SNAPSHOT_B64 = "..."
works = json.loads(gzip.decompress(base64.b64decode(OPENALEX_SNAPSHOT_B64)).decode("utf-8"))
works_df = pd.DataFrame(works)
print(f"Loaded {len(works_df)} OpenAlex works from the bundled snapshot.")
display(works_df[["title", "year", "cited_by_count", "source"]].head(10))

### Heterogeneous knowledge-graph construction

| Decision / 决策 | This cell / 本单元 |
|---|---|
| Idiom / 视觉表达 | Heterogeneous knowledge-graph construction |
| Task / 任务 | Inspect, compare, or explain the stated evidence. |
| Algorithm / 算法 | Inspect the named implementation and parameters in the following cell. |
| Data and units / 数据与单位 | Use the adjacent source note, units, sample, exclusions, and baseline. |
| Evidence boundary / 证据边界 | A display does not establish causality, consent, or community value. |

**Try and explain / 尝试并解释:** keep the baseline, change one documented parameter, and explain which reading changes. Retain the original and revised outputs.

In [ ]:
# Build paper-author-topic knowledge graph from the most cited 22 works.
selected = works_df.sort_values("cited_by_count", ascending=False).head(22).copy()
KG = nx.Graph()
for _, work in selected.iterrows():
    pid = work["id"]
    KG.add_node(pid, label=work["title"], kind="paper", year=work["year"], citations=work["cited_by_count"])
    for author in work["authors"][:5]:
        aid = f"author::{author}"
        KG.add_node(aid, label=author, kind="author")
        KG.add_edge(pid, aid, relation="authored_by")
    for topic in work["topics"][:3]:
        tid = f"topic::{topic}"
        KG.add_node(tid, label=topic, kind="topic")
        KG.add_edge(pid, tid, relation="has_topic")

corpus = (selected["title"] + " " + selected["abstract"].fillna("")).tolist()
vectors = TfidfVectorizer(stop_words="english", max_features=1200, ngram_range=(1, 2)).fit_transform(corpus)
similarity = cosine_similarity(vectors)
ids = selected["id"].tolist()
candidate_edges = []
for i in range(len(ids)):
    for j in range(i + 1, len(ids)):
        candidate_edges.append((similarity[i, j], ids[i], ids[j]))
for score, u, v in sorted(candidate_edges, reverse=True)[:28]:
    if score > 0.08:
        KG.add_edge(u, v, relation="text_similarity", weight=float(score))
print({"nodes": KG.number_of_nodes(), "edges": KG.number_of_edges(), "node_types": Counter(nx.get_node_attributes(KG, "kind").values())})

### Heterogeneous node-link diagram

| Decision / 决策 | This cell / 本单元 |
|---|---|
| Idiom / 视觉表达 | Heterogeneous node-link diagram |
| Task / 任务 | Inspect, compare, or explain the stated evidence. |
| Algorithm / 算法 | Inspect the named implementation and parameters in the following cell. |
| Data and units / 数据与单位 | Use the adjacent source note, units, sample, exclusions, and baseline. |
| Evidence boundary / 证据边界 | A display does not establish causality, consent, or community value. |

**Try and explain / 尝试并解释:** keep the baseline, change one documented parameter, and explain which reading changes. Retain the original and revised outputs.

In [ ]:
# Readable overview: label topics, top authors, and top papers only.
pos_kg = nx.spring_layout(KG, seed=301, k=0.65, iterations=150)
kinds = nx.get_node_attributes(KG, "kind")
colors = {"paper": TEAL, "author": PURPLE, "topic": GOLD}
sizes = {n: (210 + 3 * KG.nodes[n].get("citations", 0)) if kinds[n] == "paper" else 85 for n in KG}
fig, ax = plt.subplots(figsize=(13, 10))
nx.draw_networkx_edges(KG, pos_kg, ax=ax, edge_color="#D0D7D9", alpha=.45, width=.7)
nx.draw_networkx_nodes(KG, pos_kg, ax=ax, node_color=[colors[kinds[n]] for n in KG],
                       node_size=[min(sizes[n], 900) for n in KG], alpha=.9, linewidths=.5, edgecolors="white")
ranked = sorted(KG.degree(), key=lambda x: x[1], reverse=True)[:14]
label_nodes = {n for n, _ in ranked} | {n for n in KG if kinds[n] == "topic"}
labels = {n: textwrap.shorten(KG.nodes[n]["label"], width=32, placeholder="...") for n in label_nodes}
nx.draw_networkx_labels(KG, pos_kg, labels, font_size=7, ax=ax)
ax.set_title("OpenAlex knowledge graph: papers, authors, topics, and NLP similarity", loc="left", fontsize=16, fontweight="bold", color=NAVY)
ax.text(0, -0.03, "Real metadata snapshot. TF-IDF similarity supports discovery, not causal or citation claims.", transform=ax.transAxes, fontsize=9, color="#4F646B")
ax.axis("off")
fig.tight_layout()
fig.savefig(OUTPUT / "openalex_knowledge_graph.png", bbox_inches="tight")
plt.show()

### Sparse-graph PageRank comparison

| Decision / 决策 | This cell / 本单元 |
|---|---|
| Idiom / 视觉表达 | Sparse-graph PageRank comparison |
| Task / 任务 | Inspect, compare, or explain the stated evidence. |
| Algorithm / 算法 | Inspect the named implementation and parameters in the following cell. |
| Data and units / 数据与单位 | Use the adjacent source note, units, sample, exclusions, and baseline. |
| Evidence boundary / 证据边界 | A display does not establish causality, consent, or community value. |

**Try and explain / 尝试并解释:** keep the baseline, change one documented parameter, and explain which reading changes. Retain the original and revised outputs.

In [ ]:
# Optional package extension: Sparse-graph PageRank comparison
if RUN_PACKAGE_EXTENSIONS:
    try:
        # scikit-network: sparse-graph PageRank route for larger graphs.
        from scipy import sparse
        from sknetwork.ranking import PageRank
        node_order = list(KG.nodes())
        sparse_adj = nx.to_scipy_sparse_array(KG, nodelist=node_order, dtype=float, format="csr")
        scores = PageRank().fit_predict(sparse.csr_matrix(sparse_adj))
        skn_rank = pd.DataFrame({"node": node_order, "label": [KG.nodes[n]["label"] for n in node_order], "kind": [kinds[n] for n in node_order], "pagerank": scores})
        display(skn_rank.sort_values("pagerank", ascending=False).head(12))
        _extension_status.setdefault('Sparse-graph PageRank comparison','completed')
    except Exception as exc:
        _extension_status['Sparse-graph PageRank comparison']='unavailable: '+type(exc).__name__
        print('Extension unavailable:',str(exc)[:240])
else:
    _extension_status['Sparse-graph PageRank comparison']='not run (optional)'
    print('Sparse-graph PageRank comparison','not run; enable extensions and install its package to explore.')

### Optional live refresh

OpenAlex access rules can change. Keep the bundled snapshot for reproducibility. If you refresh, record the query URL, date, result count, and any API key requirement in your project provenance file.

### Optional live refresh

| Decision / 决策 | This cell / 本单元 |
|---|---|
| Idiom / 视觉表达 | Optional live refresh |
| Task / 任务 | Inspect, compare, or explain the stated evidence. |
| Algorithm / 算法 | Inspect the named implementation and parameters in the following cell. |
| Data and units / 数据与单位 | Use the adjacent source note, units, sample, exclusions, and baseline. |
| Evidence boundary / 证据边界 | A display does not establish causality, consent, or community value. |

**Try and explain / 尝试并解释:** keep the baseline, change one documented parameter, and explain which reading changes. Retain the original and revised outputs.

In [ ]:
# OPTIONAL: uncomment to refresh from OpenAlex.
# import requests
# url = "https://api.openalex.org/works?search=network%20visualization%20knowledge%20graph&per-page=40"
# live = requests.get(url, timeout=60).json()
# print(len(live.get("results", [])))
print("Optional refresh left off so the executed notebook remains reproducible.")

## Final-project transfer task

Add one paragraph and one figure to your team claim:

- **Network question and task:** Who or what are the nodes and edges? What should the intended user find, compare, trace, or explain?
- **Idiom:** node-link, matrix, hierarchy, Sankey/alluvial, arc, chord, or small multiples. Why is it appropriate?
- **Algorithm:** layout, centrality, community, path, similarity, or temporal comparison. What parameter or instability matters?
- **Evidence boundary:** What does the graph omit? What relationship is inferred rather than observed?
- **MVP addition:** What will be demonstrable in Week 5 and tested in Week 6?

**Required submission evidence:** saved output, caption with claim and boundary, code cell or function, source and license note, accessibility fallback, and one next test.

## Selected references and projects

- NetworkX documentation: https://networkx.org/documentation/stable/reference/algorithms/
- OpenAlex API reference: https://help.openalex.org/api/
- Beck, Burch, Diehl, and Weiskopf. *A Taxonomy and Survey of Dynamic Graph Visualization*. Computer Graphics Forum 36(1), 2017. https://doi.org/10.1111/cgf.12791
- Nobre et al. *The State of the Art in Visualizing Multivariate Networks*. Computer Graphics Forum 38(3), 2019. https://doi.org/10.1111/cgf.13728
- Ilievski et al. *KGTK: A Toolkit for Large Knowledge Graph Manipulation and Analysis*. ISWC 2020. https://doi.org/10.1007/978-3-030-62466-8_16
- PyVis: https://pyvis.readthedocs.io/  |  Graphviz: https://graphviz.org/  |  HoloViews: https://holoviews.org/  |  Datashader: https://datashader.org/

## Atlas Studio enhancement · four levels, four validation tests

**Domain → Data/Task → Idiom → Algorithm. Validation belongs at every level; it is not a fifth level.**

Keep the original package experiments below/above as optional depth. The new cells connect them to a real communication or community question. The self-contained snapshots are embedded, so downloading just this notebook is enough for the new data examples. No live API call is required.

| Level | Decision | Validation evidence | Project addition |
|---|---|---|---|
| Domain | Audience/community, need and task | Observation/interview; whose need is not established? | Refined shared research question |
| Data/Task | Unit, relationship, time, coverage | Source record checks; domain-informed abstraction review | Data dictionary and provenance |
| Idiom | Basic baseline + advanced view | Observed task comparison, errors, accessibility | Justified linked-view design |
| Algorithm | Transformation, parameters and computation | Tests, reproducibility, sensitivity | Working MVP and evidence record |

**Week 4 → Week 5 → Week 6 → Week 7:** individual evidence and Reflection 1 → proposal plus MVP → demonstration/user study/feedback/response (15%) → future-roadmap roundtable (5%) and Reflection 2. These 15% and 5% are the split of the existing 20% component, not extra grading weight.

### Scientific communication of emerging technology

**Domain:** A museum educator wants to explain geospatial AI. What misunderstanding should the communication prevent? Ask an audience member; do not treat this hypothetical persona as validated need.

**Data/Task:** The new snapshot contains 60 OpenAlex results for “geospatial artificial intelligence,” filtered from 2019. Every link below means **provider-assigned topic**, not citation, causal claim or a verified NLP relation. Topic score is not our accuracy estimate.

Source: https://api.openalex.org/works?search=geospatial%20artificial%20intelligence&filter=from_publication_date:2019-01-01&per-page=60. Retrieved 2026-09-14. Metadata CC0; sample is not a systematic review.


### Paper–topic source table

| Decision / 决策 | This cell / 本单元 |
|---|---|
| Idiom / 视觉表达 | Paper–topic source table |
| Task / 任务 | Inspect, compare, or explain the stated evidence. |
| Algorithm / 算法 | Inspect the named implementation and parameters in the following cell. |
| Data and units / 数据与单位 | Use the adjacent source note, units, sample, exclusions, and baseline. |
| Evidence boundary / 证据边界 | A display does not establish causality, consent, or community value. |

**Try and explain / 尝试并解释:** keep the baseline, change one documented parameter, and explain which reading changes. Retain the original and revised outputs.

In [ ]:
import base64, gzip, json
import networkx as nx, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
emerging_papers = json.loads(gzip.decompress(base64.b64decode('H4sIAHJwqGoC/8193XLbSJbmfT9FRt2MHSFKADIBJHyzQUuyJVuStaKqXO6OjokUmCLTAgEWfiTTExvRe7W3HbFP0DsTMW8wV33n3RfpJ9lzEiAJkgABUlSxK8pVFkUSicQ5ef6+850//YGQf4M/hPyk+j+9IT99ZtSzDdt2qPfTQf6LVKWBxN+9i7KwL1IVheQy6ssgIfdRTN7LUMYiUEkKf42SMbxBBKQbp+pe+fjX8zCVQaAGMvTl9DsnUsTwlZZhseKVLA7wGsM0HSdvjo76kTqM4sGRaRxaJvWOkiQODxk3mGHMFuarVK8mgQ+aljFdbjRWPr70J/3z9P5K93hrGq7tFt+iXw/EndSXhzsYxGI8VD4sG+5ulN9ub5KkcpSQXpr1lUzKH038KMbdMQ49z7UN+MeinJnc5sV7/sdB/TpM03Aq13EiUkEuRSgGciTDlIiwT7rBIIpVOhzVXh93xzAdaricGm2uTw2ndh+CaKB8eHz6UatwkK8hFMEkUXUr4NTyYBsMx3I4NacL0P//c/F4HuTkKYr75Qf0U0luxPIFfirJF64gGUuUKvxR/7X0NMoC+ko+qj5K3Ov5r4+j0ThLZUwS+NhcGEmx3SuvwrLw+YMwLdyByNJhFC/cwIdoKMJQJuSDeMju1MP8O3rZV5WSm2hSWsYwVgm5HipQivH0u/9QPKhlZeQec5lJ6YoynsFm3EXRA4nut9I72qh3oFLmkedy0zAoiBOzaJXiuWzfescZRbk3TNuyucu93YhdvbCUtlitbvE6adK3WJYEEafTF5tErBeBCr4X0fzTX0ApvypJzrL5a59l+CRDcqHWixUzHIdR1zBWxOo2ehKwTSQdShLKbykZaPXTT2NLSbObJQ0OraOvh1/F4BDfDz8z6vAqaePW/k95x8MzzjNMz3TsNoesxa3KZXSzFA8Y2YfzQfT16fY2U0EfD9vTb2ksfFzWrpaxY1vDOFyec9NlcEjZuzrqpwduW61p0LkbOYAdhGtNf9OkZWBsBv4QNOhSqLKm/ZaF5FdV+uZflYiepCIflGg4wR2XmtRk5oqqdfuPAtaUwCkCWhb5+sDJJREfwaC0U9UnDnkFu9M9f63dsJEYj0FwtjroF9SPws9w3huV6udtoH4vJPfc1XLvcNhY12wh+C/l7KECurbnuMxwrVYK6LDKddzIUZRK0pNhMvWzLtRJ94Z0x+MAfLB892uWYeEyuOFS23U5340eNrlD67SztKmqtKmJ3tT5266HkVbi0UimcbP1+wLW7ztsUTSTcHjxUsQDEcuUfBTBj7/5yYMo/y6VEdjL//e3sYijR/V//9d6NbWowWzwIFbVtNnkTfVwVffMFrrn0aPoLkJHC45ykG7wZayOYVmV9s9x92//uImWx6PUYtbvH+QwZmKMZVHThlPgJeX9GR5ghar0iqsvfKC109cQJLgmd23TWpHd02/jQKhQ3AWSFLcfTIj8hseKStuYGLBNWXwnwnzXSre0mYUxXffIonhQQYhIIbJgDq+M4x2+gYCblFWfqLexuIe7Idex7Ct/ZlJLsnYr/WEIZr1ezGGtIGcOMx04WJ02cg6fqFzNWTYSmDK5U4FKJ/n5HuXHeuetSGRjWO1xhuvxbMOzubVHn5NhoGWBCkIAYe3I6cOTNB7DMS7y/WmlgEvCiNfAA61GpZvU7Bq8vkkmMG4qxVJf4IU/QlxfsjjvYIVj8lYFX6X/0BBjUcNzbBNscm2MJcBxmyUsRphm0Z5cG8fvUSX4obEYy7jC7lgtNJLZR9S2Xc+0D6ntYMhVqY90E310zWp9vMyCVMEdYjpJ+EMVSnIBiw1R6to4OR7FAJ8asJ+228bmgJTy6pMBb2OW09qZvhkmNyuvdyXSLIbbvgA5yuDwgSMpAp9fO3rNZ9BLBFn1WnUrkgfyahxHIN0pBBTTw/L1s3Mf1Ym57YKxPH12DF9YOio+wmtwtB4Po/VaCXIEEbPLbWcLP+8NuQY/PcS3JJgMWRFl1F56Au+CvSHHQZShksMJW/rqEzUA7QrI7ZMKK0yp0cJhNNyjBJwgh4GXaHQMwwR/sVJ3+Ua6+yLRCUcB9hyLG45rtAqSLLNakWBfe1n8KCfTRRzDmaJV60yCwwiiWh8waofR8UCLuNtiDfBWXreGWTYcHKOyxLTYC4brgGPXg3t0d6PLuaT5KGk78Vs/R/HDfRA9lfQZvAW41VfgMwyl9mlLC2lW4Q8//h7DNcjJj/8KglDGDSYTolnXYhXKeZlnN3RaMoCnn/hg+bQYxODGjMQY9XGDjAnc0m9g8FOtLY+SDLWLNpiGs2/AIwM3OUWZh9/mwSvePInh+JJP25SxFrMsDLMsLmVVeuuZm+jty3mdHgqs4XGIRZndygpaNYrzsw4hwCsPk3EUF5Ger+3gktPXygCudcKZVRcS5BefZ9hme3MehtHjWuWFqFzHvK5hu8zbkSU+WxS6tWmV2pTMLMniJ9tUrnrqN9CFzwteLuY2yVm28NqCi7zewnITY/NqC4vZHtAiiPLlXSwWjs3lE7Um14IPbskknY8WHautzKppeEdCiySqp3FIwaWzWaV+upvY1RcyJhtnH3dd6vUczMMYHEyqC1btd88DcdtGjWSODRpP+d6LfrOP5qFZKb7V0jlelk4y86ySXIwbVfXn70LFpJsEYkTeDkWalpxiiFMjrZ5fSgHshQqfQInhNRE2GF54hsw17dV6IOreAOt/8D3gGHT8CAvvsGIsOUShSuEphYMDkgi9KXA7RSHioG0RA51meCEWnSLRBDc5joIg0+f0eJbBSZ5R2pDhI3xlUd2wTE6tSr3epLphmF613e3C+v97JmZG90yKIB3CITUWfrom24R/KDMcw3G8Nk6qaTVf/3L2hPRS3sH1fJGkayLfjbNeEI7Y1etIR/DshxLPe4jiRhCgxfmOCBlHSRTUajbXZ5vlWJ7NXHN/WSYIKgeLCgt7B28scuelbz8NH1UchXh6bVBsvPjxdzCnC2b2g/CjO3IiQiWDhXMpCCS40deH5K1QgZw0FDO4Da606TjPKGagTUJve3oAgL94NunHM1uh5SnIHvHz08LB1H68AQuNbjI65cdZHOOhvmDq8cO38HI/2SZ3RalnHD2ZzGTWPFrcPouM1c5qD7q431xqP8MexMlQLqSRG0olm3vQXs1a3gVRBI6PSh7AAiRgSWaWcr6a2lVoWJjnQGDlGW3OFpN563dES0DpNCE/a0ese167BKwZcViByy3WPvit19otzfhUGl/dB5nqv17yr0cqXND3hdvVCdqgjaE+FnEQJeR9FH4XgUz+8Ze/nod+uTapHmRAjgXodHkBYTghx3E0GDZYa/Cuqctd+M8qpKCkeDfSxzu9kQmoFhzASeE63wcyt6ckCrfC87RwpqllUDC7T8NsoBL8iGF7bpWiWmz/9UxDyyaF0BFUxNmDKw3ulzZ4NmWm67XJUJUyB0tGN8eU9MnxHE2C2cZZpjlfT3OAYTjaveaIPoRD9iXrrO/A1vXLGabk9dpKar12v1yNlWoZ8Zi7qnO4HBGrJAfI9WRw3/kUD8CCf8fHfynGB2XVupI6Z3klU9i6h9xJPo46XV+nofC3Uedd9v07JiXuZawtci7H5FLC2vs6Tu6pURbkTvn7GHPrT2icpm7fm63U2myZejYxpO3A++GPB3+rdKVtZ5MU1u7NDdZvXMehzHO9NtGxZc/KX0sqXdreRS+gMT5GT9qA74UAlTrt0s7Vx8rn8sMt+/RarfUG1Z5uXEfphmOCW70JJLskNmEusmEusi1NcGnfluzpQB+SJVPe/S1T9+W3XUoIA5PfMgGxH5FxHMWNSvyLGKo+eT+MIDguhcWX0VCMRgLN3ndBPg7BHA+Gpd/3hIJLXKsYHfthk9XlHLwXYxWKfSLlmNyLBznP5f038hkLV6Xwt49nH2gebAs+zZ2qJTeOTBsRTIaDMa55aHrluGyxUcLexORatMbkTutYlxCeC62bYQLm90Smcj2CQJs62EvDBt1o44vOcS1Lpi4EyxBM5pfcwsp54KjAPwgpgnW1Cncp5+stbw/iJREWcjA/M0YifmhV9NX4Rs+1HQ6OBvN2hqxtl1E+jkAav4GDjD6jyFX1dYuU9bWMfTnWT75JV99GCKsoAdl78NN3CXuzhLY4Hmbht0ySX7NG/JPlsVJ/wkwzZ9CW61g9Cn/SuY7BHY4fp45ZT/pZLNuknVd7n179kmMwrmswGLQtBoN74C8fUseynepo1nL+CZxknueGHDgBHb5XNBTnWmkxTWS6ZjsEcE1tu0IsyjYVXS1Uk7qFWLbut3I44lN2VBTauofpbZDJMbgFaXtcI0o/RtWNVVyFqa0BuVnQ2pWGlAW8RoPGOhhsUeM5CSqtwLO0s06zxxPyLtNaiVnlXqQCcj2E/UX/8TqOQFFB4qagwO3A8jr5lGRgljwTQqNKG0s360V0qxVFLx/3A784wRioyPeUgf6rhSFdqISjhBrg/LVRDj5DYlZcH1O103op6KbSaad0Wkut9X2pDh4dh3G7lVV1zOrSUA/MZkq6g1j5Gu6R1//qHX8T3V04Tm3HcGlrheypIF3rfpYcxqm8qVzemnLAdcWekhrBLqPJhZtrVMR3cPmRHILYwxFBzqIkkSos+7sTOD0G5K3A4mhZV+UE7iV3km/Ed/GoOrfgkjS4vKCm3PAst6L78EVagZuVD8TJNo5E/E09HloUTSjIWGWnirkRUP4Fsq6uro9aLhp17rp7w+y6tsYyug5i5qnXqqJDnfUu7mISo42jzXQC2GU2BWNpt6/W1pu7tUa0JwdaDxf2JUehYtNwiqvXCrwpsrHIEO+tH5gZiK9w57iEmUbmJ+UwSuFgTkl/FhRlOlXSXId9Q7pkVAAagymgEUxNHMGLJI0InIYZlmu1SsCKH54Bgkr8ZAqCsrlLn29DX6ZssmESaed5LJfq7I0D0TK1W/XavFwti3GNwWY6pWXtKCF8Vojrq8FKpFmR+M0f5miSDp+D2GjMB2tbeXlIzg5J7xAzR3DqlWxsN5DfYA8xbIzCH/8pgscyBcGliJUgH6If/x6RdyLw8f/gXjyKpoqtaXGIYe1tHGI0vaciXsiFRPda0ArShAtdrVWx3KreanLbsI4ekuTw0VUetqwx99k2l1KXrkEDqXAao2/RMAB+p6eFlTtwlfYplPXiWC/GF+KJSIyp/cUD5Sfc8ns1GC4gLip7mtEGwLeH2sIkbVoiu8FEgIjFE4SxkmsEzwzFOAH5K6EeYvDIyFtEDaSRiLOZ6aozLyZDyIXLK+gm4rBoW9ggRNOSueAXn44hpB0pvb1ocq5EHOcQ3JsaZK3dFhJv2C42TtodePCe26mE7pmbQHwY9apFNIf0HJRv7YAcR4NQ6TvpDmrjM8oMLZkUE7QOc1u1luwQaGRZPOcNMBxwxkzG94U0MpmlC4xIWuNAvLYjQPyNHGMqJ8xh3SixZAjOqg6fS6Hat3GURCO59IoOL2deQl1kJ0sSvOQd+lnOlNGou6BNCo3L+aRc9+hJNVDko4hFIh4W6AcaCT0oxNmu53irkVoXbMEj5rZyVADsXq9BedNyBgw12McHPA7Ao50+3r58lEE0Lm3TholPwztSvvT91HYc25jSDmD123Mq9Zbtu5XFzFFpnsst02CtoOgv2J1qavyvZznw2I1WHXEvE10anq5WUINCVOK+eIdadUKzTTv2ssHdEHt7Ju4TQT5l/jAuAXm6d304LAQWGN9K0I9BlMVlRzEQkpyCfVNgOBs7X1wIYhyjBl6Qm0j00CcaZbCU9pxqKgL2fpEDWTQ5TBPp5EmBYWjT4JYHJNfgYhdC2fPVg0o72vnaKuBj4NQeWZRxDiHMkfrqy86jaSprbBqVDAfU3AiEW9MttqTpKiznEGtrHNoeOY6DjHPevhg9uF6Hi6A9CD1bBX3cq4bg9nKCNHKCHqW6y2ZHzXGgUOXJMRbcZO2Jp8ncKGwyRO3tczaF9GAjqSb40NDxov319bMRfDPRb06ttu+C+UX5D7BU0j3szT/+i+oPwcW+EA/JcATmunTBQ1StsN+o0xAp29YqCvc0hH33cV9gczBjiu0BOjGlHtHWFhD6XG3bpW4Gs+qepnhYxGx0niQGIXDEo1VP8j62QYz+PApEkeTZVVMbc56N9LPM+iLmIpZ8CrlrKotsjq+vs5eXxSMrNXLrhYDv21+zACtXJoQYeqbVqmi5y7qQa+tOXAbxMIXdba/MVYLZNixeFbYqniAtsJ+nEnqz5r2TSpu/hE9qRN1n0W9qqePtg5Lky8IreHTLkHzOmvKxhu0hMUE99OBE6AagUvoRrdG9hoSAeoM1h+M5byw4hZAlTcjdhFyAgx2LQRFuC6Uj6gbr/fpNbQzdrMgUGxuPnr5COJ5rMjrkh1hof3YYjZ101YIsVTKaJyVP4Xgb/pYhjmptKpK5LkLrwPiCebTAXLfxfnlNEF1lqfE/Pyey9uqOhh9YzIMjgrP9+LwzxJUMQUqkjJsyTKel9zWpyK1QTxHmP7uDO/G1dNUrOHpJ9w7M6kR2rrAYrO6ichYLLtHPyAcxAq/j9HtjfdGihmsb3irpVi9neSXgOWcSgerNVnC79jCNaDUghurA+3XmiHYq7RffP/Odpavqhm1zBzyL/THfWZaGvWBp0bD2jAWiRp7OYhA7zQF8OwfLVcHQ29NQrij370z2alHbcc2KdpG5TQIzFEYqkfPWyw54haEEbZMiKaqIItTcYVNGsRbNnbvwLWX4CFJS+JeWRZ1KkJy5EeLc8aoV9Ervwen9Pbg8Sds64abkRC9zPDi2o7EvLrU8sx3XilPTkA1nAiKi48zXkJtLgc80FNP2+3muty5yNJyquuVzuU4KwSyZo0WJXf7FK4TPRa9bQ3Q2YE2oxdOUckNRMBjCGTZCtA7Czv/xl79eKREvpHaRauEO3LsvjRx/HjeZyc0Kfsrr2wOIr+OBnEXZea9aQl5dXFwmr4tG7JzG+VGu4UbBkk3xUbVgcoutekPErrhQELLu2prefOpv2hS0u1K1TXuTqo1VY2zeqoFGcy5CPNshxblrYLmC27BEhxvu798lVridEJZybnLakpCwBlWHj1MLQdEnNl3CEhNDzUrA5caTxnJdcElsvkuafy2gOmsk4srfafHclJN2IzLAKkaUW02IUsmQ0lBPtWzLsy32HJzreUI+Iw54yn80bb04UXi+bVcwBfWzmIOAAyunXId1utSs9Hy9/aPPq3kn9+NxOhrJZheWthUsyPXWgFuPVTo9h1r0c9k0T1Kb3KaGtaOq6VSi+mWJeiFe3JbjDtpqmIWNRJZj8Ipe6JZsQjPqks6dFgCJeQgS3WFHQC6l/qy0n1s/tI6xTMagHxKhcceffjk/6ZjeVuCa3MsF106gyxWhOlqHhn0IAlZpDDciyOWmVwuy6ZzEWCPH9BO4+DKn11NBIMqP7JnAbiwKVi5gumUgc2IQRlikytaD4RxDlxotgzPboK0yqMw01l+8VMvX9jhZewzZGlvvMtO2meXsKu+ztt+qWgN7Ks0KGrMnEWuYw1pu3C4K+4S8guM3iJII9O91ixD0XgmI/7qpLMFXjwuYUkSuZQKXTtQC9M2PyGkGF2usdlLGKHVWI9KfUziXv7dDrKLmIfOfjwIbgyAl2IedH6Z+HGnoh0bExHoKRLKdo0o5sm3aHu/A+zs2eGGsM6lurNzETtp2tWv2PoB4AMkh9D2dh37eAVTUYmMpw3VFDg0595gDxsFp01wJy6iORI9BEGOIgedLmV08dxSb+jxdnSOyLMM1aKtyx66PKkdbSrAOcFqwPSVpC0SWL2K58YCEG5lEWYxPPggKB6UZEjgcgf16J+AMK/nIH2UYSljGySE5VdifdSeyUUWPdDfA3C7YQNVMN+8w6njPcGrfrFD/FXt1PN+rjcwo6JNnHlmWyztwtnzDEULVAEBjj+aT6xDOMBkmi8wXHJpz3tNaOs7uYI+LY3DzER3Vn+5lSaozgDkjT5NMfiQ90hNDWHe5TiDlA1wUsW7psNwecRlloXqAGJT04LhpMRsNtJxXkdbFUT/LoVVgJZ4zwmObOWmdygzl3kd3cHBbNCeFjROJzP2cietruTXcinuak+ZC+G5zSpnTNLypDclhvABTwdT4ctKpJHZO28DBj5JYPmLU4GA5y6uOGvi+Zc8yTT02hnPbtj2Dt/LfnZrO1Lxo21nkfz0ORJLg5ot6v8TMp6bBf1ELONtbW54BFzfydD0OsgKfbTfa2IMoZoUr7Sa6gzNbg6FfyccoT5a/bpd6a+D/nmfIYrgErDXFEnDSnEE/DZRICJjQNBEr7TO3sRo8iAY6FsrgMbrGakIcGVDyTGbRC65jrCRtQ4tU9I3r199jt9Gfcigu3jWcdCOVplImf96KpgVBz6NBnOQULVQP2a0O8em+093gPWtL4WCiyWqXcXtJnhZMNes6LwI5WjXT7tp1M3XajyOppuUZ7VvLjz9d3Z72blv3rWpi3ftsEf/0bunnlja3SQevRATO0ZfoIZqIMpNLH6GQ7yEqUOXw/iZCgsKzH/8RJv6wcbwoNwxmeQ0VZ7HAXjYbrkH+iIXnnzdpNuqA73awzAv+6qb3OmdYQ4f4Noryutal+Crjvsjp6O90I3s8IW8h7oFz7gqPMNK9j0EiX2+TWy/ISSHsBW/QfDZ46iWIyezSsFyzVQPQjtGYbs5xVDSls3201tJ8jKRL4ZnaJt1RKFZFelY1C64orcOXYLhd0uyc6i2exv6j1VJcQ/TWPEgxQd79iegvZO5SPS0R1S/IfL/0q48igi+BNV356q4xMcCp43oVdBHXCAVH/wOFxB8i82k4kEm7oR3bGVuIDTGtPvALe6tZSJlVCWdk/wzzNpCXPHdOIZQE58baF8DLcIvWPIRKG14r/WSUueu9EJ01LOpMp34ETkHN0A+DevnlXYxXqfWiM013Usvatqnoj2fdq/cER5vCuV0uN/8RfgQtiZYJ0W4/wfuxMuYPo+bSs+F6tkXNZ7by5tleOV6ccoUY2SRQfeQHR9ByBAagOPPfkGlsfID9JYWq53b4dCRjDWo+UbGsY/VvaWPjBNm/zNIor2dk42l1lWp2m/nJFctAx3tD8R0H+O0uDe/UdAoex5MZh35Rm9ILKRUna13mfD6GZbn27zAeY7ZRzV05m/XwtaoTH4O+VOV2ysBk0fmSkbNkPZ7fgggDI6oKE9a9uT1/d3583r0g5+DSX1ycvz+9Oj4l3asT8v70U++6e4u/6151L770wN08vyIn571u7/b0hlzCq+9PL0+vblekfeYMrkltgy07Usk4TjpYJITIJel8C5Tq0M4T70Bg0sGvqeylc/YeSHqa25ozBl6W0WqgBHXsmqrtAqV1zuuWr6PoOGsT17oaxuV6nIKH7uwNQ81cqjHUHOIkau+pPNWr/SZ9TuuacZUHqul0ph8ir0odCM1lZQi+IPQ6bxzIjYTbruGa3KjoDShGKeJNLUFVS9MqWve8dotnpjGNFyrFXBh+V10zjdm2v10liU3BQOUOqGdTalX3vG6SnWWsxlaVhfAGx2zJx2JMR4lRpU4cDUurhetZnsNbYZhfkL7FsxydHLWR6Qqu1J7Hr8y1rBYko2UCBs+6fpnxswXbfEUap7QOxBQ1R2VZfyhIT6YTQa6yQRwNy+R9GRxuEQG3LYlILxuDS/gg0kbgoYujwSqGDw8lOZ0mgQtGiI2Z+lpysnhYje0Y1Oxwl1lmx/7XSnwv3TvAEOLWPBvguZ7ZakboboeZUUY3JIt/OYAjpch7BtvgYW2CvliRei2138Ylwo0Sn9WDhb+o37IQwiu5iPN9kop8UKIJ0cRc5jgVQw9PKxtmOnWFwtGcTROxvjJJVSHJoKrXl9ahTSB4xskuOdUMecSKif5bP9P0O340GuWI7jhLhmQYZbFOe2JTnwif0YgzH6KGfTigKtUAC3OjPu+XGWKmQx9qWdwzLfZ7cyxtwRy4+yFqcOMaqMmph9zou9Hi05k0llgSZvL3SuNXMbHTvuWmzSHQpM3XUnW+KPI5WhjdlMnJP/7y1w+g5L2sTDWvhvAyMssjsK2vQOUummjRKIipCQ6psRXiCaxRhscuqHlEwMgj6Hs8xArHiW6wK/lDWu1BtgJtOlDg/6Un5UORg8HdnfwLqvLPA3hFbOeZOoxjXvSr1F6pYRugzOazYf/zWXNLZAlR8lsG2tu5i2K4135ecCvywBHCdIK6SM3R2VATwyOn1XxRY95IvLQIEeiy8nQUVjF0Z+3VLW2QLZz2axntYfb6kbYdkbJq5n4BvwxkRyLz9KAyz1/uhxvrGUALhcENgDJvZfhVjHDy7CHRYwdLxlSGENhdRAswQYRqCfJR9PvZKHsQSROjpS7aGhWMRl2U9HexGEnka3hDTrCPPMmj+mngKdHcdc/xKcHJEGUJZiwPyCed95Jx5zxEMlp8ikEgxomcDlLK+QRyvnec9QxR4iSEu/S3mh7KQAqOAoXyfzhQxRGVk5Wx6so9tiFvojeGVU2NcIsfnrWH1YJsc0Q8kr0zo1VDuMmrVeRK5OWoi2kHYYndpJGMzGPa3rqI+uWtKKapY1VX528RC38jRV+U/NeegiOxAdnjcT0qHLYEc8KmswOG6VuRPGgWo6/ST0uJkNcLac9Sw2Wp/7TYzllDZtX4357ErtpdJEFv1GMEmpkNZCzCMtDyNFDgelb+CrXwgLw6DkTWl6QXIUaYsEPndYOj6xhIbO5aqzU+GRdMMbLlaEFyDecXnmy5v6vpuWEjcaAofjCCwG0cxUWmJ5Fp8lx0JnKsWbyaGXT/M1ZMT1ODghmzKcTuL50TrAjJnk2v1zgZ85D8ckg+xioZhuIBItV4mQLzSabiUZD3qDiBKpNxgbUj13AcNA2/pWC8LZfz1eaSm9Pbm+7x7ekJwWT+8cXpG7JRv8nGIE7WloDEgtBR95eYps1ZtYC6eycg4XqcrelR27N5q1z+jke+awpyC/xBahpsH0Pvua07wEwMqQy6IwT1W4Uz8Qq1qx0qXcqytx8u0ibt0swzrWKcntcbghN9X1LJM9FX8AyHfTVG3sumbD5jrlsxPO9S+XE0I5dFFevLRA30T4/C98F+5T2YAmJueF3nMnAS/QwkA5YBY6WkoMFr0OKiyfN+6n9uN3FPJ0Ty5eWB1KHhHc55rrZvOdl1z+IW9be6FZyPRhnEcNPG1zyIKz2i+lSEp2fKux63GKf76AXzdC8Yd2wTB7Lsio+kIv6qtbazdq5XEAOL0Z0a5B2cr3dHXHA7VGIQkW42yJI0ImcyDn/8LUTswk3kD8XCqGk4LN7DtVQG3iG4f91gJFVfLNj8oSIfo+9DMNKjZmoSSpnF12VH1tnUaZIzxNrLBP6eo05CZATUP4UqxbmhIemr6BuCTmpSoKU8Jx4G02uP5zzLZT99B2REmIZ/dhJ0x8lHbmigsks585i3L373LQaR4YDVxiSoRhEtGMb32EewGOSvnkC6nkgZpYiHaZ/NyaWxFNctCWLLTE9DEnRaLtRyvNy8Ub75mY/QOH9IgjmVcCw+SvJW3AlRzhtVpUuvRVakRQc5mma9xhsu445TofE/w3kTJ6nQVKi6gjGeobrLvrJO79REhYtkzTijM52NYNuKWkH4OakCvGQYlblOblubKOyuptVpfxq0w7I42OdW+Zsd8l2zhXCzfaGv6ptLLy76qiU3F/kdFpgKpmfLAi/BQYl5Fls46wbDX887PF9JjSfVv2/En0TDPvkAISZsaYmrS8HpKgJZbn89v4vFUI2K13F4dJNWaGIY03MqpuXpSTxBpBHQBW5Gp8BF4BdOxCZkzz76PcmUDH4sxpnADNOj2poGUysKJqmSQlXgX2Y/u5OQ1vhxmNadAhgPCARAfTip4gSEIU/kzrylZvY9RzuXsGhsJ2hTubfmUcjioj7htFARjEhJtHT6cbHheVes7cyowbkV0nLqz6lxy/u1GS9Rs42T9/p6RQ3PX/JgXy/EqbPHtIj8BH/scanlYdmTLW5K+svD3wdxlC+gxcijd/AMBBb04omApyRGoAElp/VCJoECg7dg2U5xVJdAHzgIVBNomnLHcGxme5Vl/Nz9OYWTdmG4QtFpUm3NDsp50IqcYIGL/gZGsq/Zr/MjEd3in+M7vFmc3eLLZOs5KxCgwambVy0oWG6HVmJFmf3P0ASxDU/6vM1qcS2/IPvjQniYk2zFwteT2C8lCFi/fgxDzrfnOrZjtzPOTk2yaSoxOXcmziOUBaNUvTabmuIayYYMe1cgnProdC6xODrSzwebJ9kYZ/yu8EK8UzLo15nlGiIUFIt//OV/z04XtGjTZGNzR/9IKCWHoMIPQ5xvBk60HKlNqTZ7UdS5VODdRk1tTCbCJlzHrOCVSDAmTfJafslWDyXcDgYGEqUebHIw649IssSX41QVGpFnn5CWAnzj3ln3uvPr+7dRBEdjTWhKtwhNKWi+S+3KIX0Os/bTGWG24yOqvhwO0iOyoOWF3QMFysWyPlObj6tk4Fw7Xqva6AvMEK3BFTaqcUU7RQvyzEB9n5ZNQbqi8nDqXpWglpzp6ZD0warlvgC/NDfajVFnFk7Ucs9S5Yu/ghkfBMjUcSmaWok9CpGmU+FU68fSW1SvxSkLqGO3Ep0beHzgoeRNvgdgh/tKgCMdktlI4xbVyu50GG5pOMMCFr2brUwd3maOoQXCks8xpCA/HevZXviLDMfdGGL3YpNpPVv3WyGogBqsHXi3hhh3cS0ncZTBcd/oshg6jHcYZ3Ds7KhuWzXrdgVWPre6MQYoA7ldO3FL0mvsAhThV+2ThuWSrYizMWhA0jTsweU4XpRVMHXeqbtARSPsWpjSSi/3BusdKqxNXTI5mDdvgPaLlTzFlrMhuJ4NYRrc9jpPlbONnH1kjzgWNVzLM22Heez3zh5xqhn+qENdm1mtBplallPrS+DAlTwUmlr25chnpUlEq93KAp7LnxONs6QqrdWa2KqBVW3z+qsm9fso4AujrOzPipGKyWlAPogf/+d+YezvmUgSUNPjoQBzFWWiETFhchtOcrNCM/MmqE1oON7MEawz4PpKExC8nIe310Xld5uaDESKGtrT8VgHwkXaoQ7rmP9Kn9/j/3KTO12Wt3q43DJMq5Xavliw7Th5jIvNHpa5Qc5oyyizDZCoHSRpA8QR+K4KOTGz742ktCZz3Co6wc204PRTpRZ8qtCCT7vQApuDtB59HanDR6pMMFvVcd8/jfznxCse0p5R19yr/GMKFuUfZzsYO4IFNJMZ1uvHUoFMNH3VVKCCQqBeVkEYcqoytwL62bLO358xgOGjmg65hBtNRiIISM5ldBdplgPwQWa87ETpIbeT7WMqzM2ZOqbCCQJO59ka8iIx1caV+xeZP1uwryz3/x38/mjDfN4Jw+PI2RU/Yo1MrcHQVQycBR81jb6p5VpG+1Cqi8gVci2yEma8F6uvCNUOv6pAlF/PxqDd8OZ0KB4aUXbUw+GIqyasWi0PCvcUoy4Q2aIakVQVvLYByOEk+STyc4AcjndnldV3MAmbqB7n1fW703Q4JYQoGBwLpEzeS1JbBdfYEDjWmOtY7Trfa9L8s8ilX8BRitRoAvso69Fplq2pBW3bYCb8ad/tXl2XRKhZcUGdpJcxbEVSVc9bKchPx0L7i2Oh1xI+lPLr9ylOI4BjfhxEE33QtZi7nIJheC8C8b1k1tQgkwH58T9hEQE51m0SURnAgnbi8yE5Bqshmsvy3GGuZVVAT9E7UHhHvbFIUF6uZZzd4TeSXjbM8EfsdIIfT3ACO/z/YwTe6Efdgk8uYRsGWSge0H/D6t2DfsvLck3bjmWaR3cZPIzDRxPcPVrdiLRJMe9FqRyYk6cJTAfC9VYjQ6hRY00+z7gOL1rZU2aYmmzdMjyXua0MmW1X4xVO+342JZ6ZseBMpjXcOr3WqDOs39kuZzuKrhqgZO+mpfRGs1bw55SSBrPc56tS5qzFoBB4GBNwL54WQGZDOIIEOR7qeSHwwNJMNqJnKIQ7ll3F5q45XVDiLqKnzjFWywqm1Dks8YDMbXvBRHYw5aRsQ/qLnPGik4djCHycjRssBTtbFeEtk3pHSRKHh4xCuOF6z+0+eqmm+U0npO+6aV5bQYeBslitsvgv0DTv6KZ5zCwiDmHHLmezTuLGVUyzbND5+WDMxtHnP/4OirpYnsPOX7BvoZKlbdFkv3JCUI0DNS5n9v/w5/8PiWUTWTTSAAA=')))
print('Real OpenAlex records:', len(emerging_papers))
display(pd.DataFrame([{'id':p['id'], 'year':p['year'], 'title':p['title'], 'url':p['url']} for p in emerging_papers]).head(5))

### Idiom comparison: count chart versus paper–topic graph

Keep the first 20 papers fixed. Compare the basic topic-count bar chart with a bipartite network. Ask a peer to identify a topic with several supporting papers, locate an original source, and explain what the graph cannot establish. Record errors and confidence; do not fill in fictional user-study results.


### Topic-count baseline and node-link diagram

| Decision / 决策 | This cell / 本单元 |
|---|---|
| Idiom / 视觉表达 | Topic-count baseline and node-link diagram |
| Task / 任务 | Inspect, compare, or explain the stated evidence. |
| Algorithm / 算法 | Inspect the named implementation and parameters in the following cell. |
| Data and units / 数据与单位 | Use the adjacent source note, units, sample, exclusions, and baseline. |
| Evidence boundary / 证据边界 | A display does not establish causality, consent, or community value. |

**Try and explain / 尝试并解释:** keep the baseline, change one documented parameter, and explain which reading changes. Retain the original and revised outputs.

In [ ]:
sample = emerging_papers[:20]
comm_graph = nx.Graph()
for p in sample:
    comm_graph.add_node(p['id'], kind='paper', label=p['title'])
    for t in p['topics']:
        comm_graph.add_node(t['id'], kind='topic', label=t['label'])
        comm_graph.add_edge(p['id'], t['id'], relation='classified under', source=p['url'])
topics = sorted((n for n,d in comm_graph.nodes(data=True) if d['kind']=='topic'), key=lambda n:comm_graph.degree(n), reverse=True)
fig, axes = plt.subplots(1,2, figsize=(15,7))
top = topics[:7]
axes[0].barh([comm_graph.nodes[n]['label'][:35] for n in top][::-1], [comm_graph.degree(n) for n in top][::-1], color='#0b8c86')
axes[0].set_title('Basic baseline: topic-assignment counts')
axes[0].set_xlabel('Direct connections in the fixed sample')
pos = nx.bipartite_layout(comm_graph, [p['id'] for p in sample])
nx.draw(comm_graph, pos, ax=axes[1], node_size=55, width=.6, alpha=.8, node_color=['#0b8c86' if d['kind']=='paper' else '#ce8440' for _,d in comm_graph.nodes(data=True)])
axes[1].set_title('Papers (cyan) ↔ topics (amber); edge = classification')
plt.tight_layout(); plt.show()

### Graph consistency tests

| Decision / 决策 | This cell / 本单元 |
|---|---|
| Idiom / 视觉表达 | Graph consistency tests |
| Task / 任务 | Inspect, compare, or explain the stated evidence. |
| Algorithm / 算法 | Inspect the named implementation and parameters in the following cell. |
| Data and units / 数据与单位 | Use the adjacent source note, units, sample, exclusions, and baseline. |
| Evidence boundary / 证据边界 | A display does not establish causality, consent, or community value. |

**Try and explain / 尝试并解释:** keep the baseline, change one documented parameter, and explain which reading changes. Retain the original and revised outputs.

In [ ]:
# Algorithm-level tests, not evidence of communication effectiveness.
rank = nx.pagerank(comm_graph, alpha=.85)
assert abs(sum(rank.values())-1) < 1e-8
assert sum(dict(comm_graph.degree()).values()) == 2*comm_graph.number_of_edges()
assert all(comm_graph.nodes[a]['kind'] != comm_graph.nodes[b]['kind'] for a,b in comm_graph.edges())
first_paper = sample[0]['id']
two_hop = nx.single_source_shortest_path_length(comm_graph, first_paper, cutoff=2)
print('Nodes:', comm_graph.number_of_nodes(), 'Edges:', comm_graph.number_of_edges())
print('Two-hop reachable nodes:', len(two_hop), '| Algorithm checks passed.')

### Innovation output: a source-grounded communication brief

1. Name the audience, decision and misconception.
2. Link one selected paper; distinguish its demonstrated finding from a hoped-for application.
3. State why the basic chart and the network are both needed.
4. For an NLP knowledge graph, attach source spans, extraction method, inferred/verified status and a human-check sample to every relation. Entity co-occurrence is not factual entailment.
5. Record a peer's actual interpretation and the change it prompted. Bring this brief and the data dictionary into the Week 5 MVP.


## Export your design and evidence / 导出设计与证据

Edit the empty fields below. Empty feedback means not collected; it is not instructor approval. Retain source records, original and revised outputs, and a next test.

In [ ]:
import json,platform,zipfile
from pathlib import Path
STUDENT_DESIGN={'audience':'','question':'','human_decisions':[],'supported_reading':'','limitation':'','actual_peer_feedback':'','next_test':''}
record={'lesson':'network','student_design':STUDENT_DESIGN,'snapshot_hashes':globals().get('SNAPSHOT_HASHES',{}),'parameters':globals().get('NETWORK_PARAMETERS',globals().get('HERITAGE_PARAMETERS',globals().get('palette_parameters',{}))),'extensions':globals().get('_extension_status',{}),'python':platform.python_version()}
Path('INFOSCI301_network_design_record.json').write_text(json.dumps(record,ensure_ascii=False,indent=2))
with zipfile.ZipFile('INFOSCI301_network_design_bundle.zip','w',zipfile.ZIP_DEFLATED) as z:
    for folder in ('lesson_outputs','outputs'):
        if Path(folder).exists():
            for p in Path(folder).glob('*'):
                if p.is_file():z.write(p)
    for p in Path('.').glob('INFOSCI301_*'):
        if p.is_file() and p.suffix!='.zip':z.write(p)
print('Exported figures and your design record. Complete your own interpretation before sharing.')